In [ ]:
# 单元格 1: 导入必要的库
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.notebook import tqdm
from IPython import display
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, MDS
from sklearn.cluster import KMeans, SpectralClustering, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.model_selection import cross_val_score

import umap
import glob
%matplotlib inline

# 设置全局字体为Arial
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# 单元格 A: 新增的检查点管理模块
import pickle
import os
import hashlib
from datetime import datetime

class CheckpointManager:
    """
    管理分析过程中的检查点，支持断点续传
    """
    def __init__(self, base_path):
        self.base_path = base_path
        self.checkpoints_dir = os.path.join(base_path, "checkpoints")
        if not os.path.exists(self.checkpoints_dir):
            os.makedirs(self.checkpoints_dir)
        
        # 初始化进度跟踪
        self.progress_file = os.path.join(self.checkpoints_dir, "progress.pkl")
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'rb') as f:
                self.progress = pickle.load(f)
        else:
            self.progress = {}
    
    def get_checkpoint_path(self, step_name, params=None):
        """获取检查点文件路径"""
        if params:
            # 创建参数的哈希值作为唯一标识
            param_str = str(sorted(params.items()))
            param_hash = hashlib.md5(param_str.encode()).hexdigest()[:8]
            filename = f"{step_name}_{param_hash}.pkl"
        else:
            filename = f"{step_name}.pkl"
        return os.path.join(self.checkpoints_dir, filename)
    
    def has_checkpoint(self, step_name, params=None):
        """检查是否存在检查点"""
        checkpoint_path = self.get_checkpoint_path(step_name, params)
        return os.path.exists(checkpoint_path)
    
    def save_checkpoint(self, step_name, data, params=None):
        """保存检查点数据"""
        checkpoint_path = self.get_checkpoint_path(step_name, params)
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(data, f)
        
        # 更新进度
        self.progress[step_name] = {
            'completed': True,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'params': params
        }
        with open(self.progress_file, 'wb') as f:
            pickle.dump(self.progress, f)
        
        return checkpoint_path
    
    def load_checkpoint(self, step_name, params=None):
        """加载检查点数据"""
        if not self.has_checkpoint(step_name, params):
            return None
        
        checkpoint_path = self.get_checkpoint_path(step_name, params)
        with open(checkpoint_path, 'rb') as f:
            data = pickle.load(f)
        
        return data
    
    def get_progress(self):
        """获取分析进度信息"""
        return self.progress
    
    def reset_progress(self):
        """重置所有进度"""
        self.progress = {}
        if os.path.exists(self.progress_file):
            os.remove(self.progress_file)

def run_with_checkpoint(checkpoint_manager, step_name, func, *args, **kwargs):
    """
    使用检查点运行函数，如果存在检查点则直接加载结果
    
    参数:
        checkpoint_manager: 检查点管理器
        step_name: 步骤名称
        func: 要运行的函数
        *args, **kwargs: 函数参数
    
    返回:
        函数结果
    """
    # 提取要传递给函数的参数和要用于检查点标识的参数
    checkpoint_params = kwargs.pop('checkpoint_params', None)
    
    # 检查是否存在检查点
    if checkpoint_manager.has_checkpoint(step_name, checkpoint_params):
        print(f"Loading checkpoint for '{step_name}'...")
        return checkpoint_manager.load_checkpoint(step_name, checkpoint_params)
    
    # 运行函数
    print(f"Running '{step_name}'...")
    result = func(*args, **kwargs)
    
    # 保存检查点
    checkpoint_manager.save_checkpoint(step_name, result, checkpoint_params)
    
    return result

In [ ]:
# 单元格 B: 修改后的单元格2 - 超参数和实验配置
# 设置随机种子，确保实验可重复性
RANDOM_SEED = 666
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 数据采样参数
SAMPLE_RATIO = 0.1  # 使用10%的数据进行分析
USE_SAMPLING = True  # 是否使用数据采样

# 数据预处理参数
APPLY_PCA = True   # 是否应用PCA降维
NORM = True        # 是否对数据进行标准化/归一化处理
PCA_VARIANCE_RATIO = 0.95  # PCA保留信息量

# 特征分组
DIFF_FEATURES = list(range(0, 15))     # 扩散特征 (0-14)
QTI_FEATURES = list(range(15, 225))    # QTI特征 (15-224)
CEST_FEATURES = list(range(225, 341))  # CEST特征 (225-340)

# 定义模型名称，用于结果保存和模型标识
MODEL_NAME = 'BrainVoxel_Separability_Analysis'

# 大类定义 (初始设置，可能需要根据实际数据调整)
DEFAULT_NUM_BIG_CLASSES = 5  # 大类数量

# 指定数据集名称
DATASET = 'BrainVoxel'

# 数据参数
FEATURE_DIM = 341  # 输入特征总维度
NUM_CLASS = 102    # 细分类别数量

# PCA参数
DIFF_PCA_COMPONENTS = 10   # 扩散特征PCA组件数
QTI_PCA_COMPONENTS = 30    # QTI特征PCA组件数
CEST_PCA_COMPONENTS = 20   # CEST特征PCA组件数

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
# 如果保存目录不存在，则创建该目录
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# 初始化检查点管理器
checkpoint_mgr = CheckpointManager(SAVE_PATH)

# 保存配置参数以便恢复
CONFIG = {
    'RANDOM_SEED': RANDOM_SEED,
    'SAMPLE_RATIO': SAMPLE_RATIO,
    'USE_SAMPLING': USE_SAMPLING,
    'APPLY_PCA': APPLY_PCA,
    'NORM': NORM,
    'PCA_VARIANCE_RATIO': PCA_VARIANCE_RATIO,
    'DIFF_FEATURES': DIFF_FEATURES,
    'QTI_FEATURES': QTI_FEATURES,
    'CEST_FEATURES': CEST_FEATURES,
    'MODEL_NAME': MODEL_NAME,
    'DATASET': DATASET,
    'FEATURE_DIM': FEATURE_DIM,
    'NUM_CLASS': NUM_CLASS
}

# 保存配置
checkpoint_mgr.save_checkpoint('config', CONFIG)

In [ ]:
# 单元格 3: 特征选择函数
def select_discriminative_features(data, labels, group_name, k=20, verbose=True, plot=True, save_path=None):
    """
    Select most discriminative features
    
    Parameters:
        data: input data features
        labels: class labels
        group_name: feature group name
        k: number of top features to select
        verbose: whether to print information
        plot: whether to plot importance distribution
        save_path: path to save the plots
        
    Returns:
        selected_features: selected features
        feature_indices: indices of selected features
        feature_scores: feature importance scores
    """
    # Use F-statistic to calculate feature importance
    k = min(k, data.shape[1])  # Ensure k doesn't exceed feature count
    selector = SelectKBest(f_classif, k=k)
    selected_features = selector.fit_transform(data, labels)
    feature_indices = selector.get_support(indices=True)
    feature_scores = selector.scores_
    
    if verbose:
        print(f"\n{group_name} Feature Group Selection Results:")
        print(f"Original feature dimension: {data.shape[1]}")
        print(f"Selected feature dimension: {selected_features.shape[1]}")
        print(f"Mean F-score: {np.mean(feature_scores):.2f}")
        print(f"Max F-score: {np.max(feature_scores):.2f}")
        
        # Print top 10 most important features
        sorted_indices = np.argsort(feature_scores)[::-1]
        print("\nTop 10 most important features:")
        for i, idx in enumerate(sorted_indices[:10]):
            print(f"  Feature {idx}: F-score = {feature_scores[idx]:.2f}")
        
    if plot:
        plt.figure(figsize=(12, 5))
        
        # Left plot: Importance distribution of all features
        plt.subplot(1, 2, 1)
        plt.bar(range(len(feature_scores)), feature_scores[sorted_indices])
        plt.title(f'{group_name} Feature Importance (All)')
        plt.xlabel('Feature Rank')
        plt.ylabel('F-score')
        plt.yscale('log')  # Logarithmic scale
        plt.grid(True)
        
        # Right plot: Importance of selected features
        plt.subplot(1, 2, 2)
        selected_scores = feature_scores[feature_indices]
        sorted_selected = np.argsort(selected_scores)[::-1]
        plt.bar(range(len(selected_scores)), selected_scores[sorted_selected])
        plt.title(f'{group_name} Feature Importance (Selected)')
        plt.xlabel('Feature Rank')
        plt.ylabel('F-score')
        plt.grid(True)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(os.path.join(save_path, f'{group_name}_feature_importance.png'))
        plt.show()

    return selected_features, feature_indices, feature_scores

In [ ]:
# 单元格 4: 特征降维与可视化函数
def advanced_feature_reduction(data, method='umap', n_components=2, labels=None, plot=True, title=None, save_path=None):
    """
    Advanced feature reduction and visualization
    
    Parameters:
        data: input data
        method: dimensionality reduction method, options: 'pca', 'tsne', 'umap', 'mds'
        n_components: dimension after reduction
        labels: class labels for visualization
        plot: whether to plot reduction results
        title: plot title
        save_path: path to save the plot
        
    Returns:
        reduced_data: reduced data
        reducer: reduction model
    """
    # Ensure data is float type
    data = data.astype(np.float32)
    
    # Apply dimensionality reduction based on specified method
    if method.lower() == 'pca':
        reducer = PCA(n_components=n_components)
        reduced_data = reducer.fit_transform(data)
        explained_var = reducer.explained_variance_ratio_
        explained_var_str = f"Explained Variance: {sum(explained_var):.2%}"
    
    elif method.lower() == 'tsne':
        reducer = TSNE(n_components=n_components, 
                      perplexity=min(30, data.shape[0] // 5), 
                      n_iter=1000, 
                      random_state=42)
        reduced_data = reducer.fit_transform(data)
        explained_var_str = ""
    
    elif method.lower() == 'umap':
        reducer = umap.UMAP(n_components=n_components,
                          n_neighbors=min(30, data.shape[0] // 5),
                          min_dist=0.1,
                          random_state=42)
        reduced_data = reducer.fit_transform(data)
        explained_var_str = ""
    
    elif method.lower() == 'mds':
        reducer = MDS(n_components=n_components, n_jobs=-1, random_state=42)
        reduced_data = reducer.fit_transform(data)
        explained_var_str = ""
    
    else:
        raise ValueError(f"Unsupported reduction method: {method}")
    
    # Visualize reduction results
    if plot and labels is not None and n_components in [2, 3]:
        plt.figure(figsize=(12, 10))
        
        if n_components == 2:
            # 2D visualization
            unique_labels = np.unique(labels)
            for label in unique_labels:
                mask = labels == label
                plt.scatter(reduced_data[mask, 0], reduced_data[mask, 1], 
                           alpha=0.6, label=f'Class {label}')
            
            plt.xlabel('Component 1')
            plt.ylabel('Component 2')
            
        else:
            # 3D visualization
            fig = plt.figure(figsize=(12, 10))
            ax = fig.add_subplot(111, projection='3d')
            
            unique_labels = np.unique(labels)
            for label in unique_labels:
                mask = labels == label
                ax.scatter(reduced_data[mask, 0], reduced_data[mask, 1], reduced_data[mask, 2],
                         alpha=0.6, label=f'Class {label}')
            
            ax.set_xlabel('Component 1')
            ax.set_ylabel('Component 2')
            ax.set_zlabel('Component 3')
        
        # Set title
        if title:
            plt.title(f'{title} ({method.upper()} projection) {explained_var_str}')
        else:
            plt.title(f'{method.upper()} projection {explained_var_str}')
        
        plt.grid(True)
        plt.legend()
        
        if save_path:
            plt.savefig(save_path)
        plt.show()
    
    return reduced_data, reducer

In [ ]:
# 单元格 C: 修改后的PCA方差分析函数 (单元格5)
def analyze_pca_variance(data, variance_threshold=0.95, plot=True, save_path=None):
    """
    Analyze PCA variance and determine optimal number of components
    
    Parameters:
        data: input data
        variance_threshold: variance to retain (e.g., 0.95 for 95%)
        plot: whether to plot variance curve
        save_path: path to save plot
        
    Returns:
        optimal_n_components: optimal number of components based on threshold
        explained_variance_ratio: explained variance ratio for each component
        cumulative_variance: cumulative explained variance
    """
    # 检查是否有检查点
    params = {
        'data_shape': data.shape,
        'variance_threshold': variance_threshold
    }
    checkpoint_result = checkpoint_mgr.load_checkpoint('pca_variance', params)
    if checkpoint_result is not None:
        print("Loading PCA variance analysis from checkpoint...")
        if plot:
            optimal_n_components, explained_variance_ratio, cumulative_variance = checkpoint_result
            
            plt.figure(figsize=(12, 6))
            
            # Plot explained variance per component
            plt.subplot(1, 2, 1)
            plt.bar(range(1, len(explained_variance_ratio)+1), explained_variance_ratio)
            plt.xlabel('Principal Component')
            plt.ylabel('Explained Variance Ratio')
            plt.title('Explained Variance per Component')
            plt.grid(True)
            
            # Plot cumulative explained variance
            plt.subplot(1, 2, 2)
            plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'o-')
            plt.axhline(y=variance_threshold, color='r', linestyle='--', 
                       label=f'Threshold ({variance_threshold:.2%})')
            plt.axvline(x=optimal_n_components, color='g', linestyle='--',
                       label=f'Optimal Components ({optimal_n_components})')
            plt.xlabel('Number of Components')
            plt.ylabel('Cumulative Explained Variance')
            plt.title('Cumulative Explained Variance')
            plt.grid(True)
            plt.legend()
            
            plt.tight_layout()
            if save_path:
                plt.savefig(os.path.join(save_path, 'pca_variance_analysis.png'))
            plt.show()
            
        print(f"Optimal number of components for {variance_threshold:.0%} variance: {optimal_n_components}")
        print(f"Total components analyzed: {len(explained_variance_ratio)}")
        return checkpoint_result
    
    # 计算部分
    # Calculate maximum number of components
    max_components = min(data.shape[0], data.shape[1])
    
    # Apply PCA with all possible components
    pca = PCA(n_components=max_components)
    pca.fit(data)
    
    # Get explained variance ratios
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    # Find optimal number of components based on threshold
    optimal_n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    if plot:
        plt.figure(figsize=(12, 6))
        
        # Plot explained variance per component
        plt.subplot(1, 2, 1)
        plt.bar(range(1, len(explained_variance_ratio)+1), explained_variance_ratio)
        plt.xlabel('Principal Component')
        plt.ylabel('Explained Variance Ratio')
        plt.title('Explained Variance per Component')
        plt.grid(True)
        
        # Plot cumulative explained variance
        plt.subplot(1, 2, 2)
        plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'o-')
        plt.axhline(y=variance_threshold, color='r', linestyle='--', 
                   label=f'Threshold ({variance_threshold:.2%})')
        plt.axvline(x=optimal_n_components, color='g', linestyle='--',
                   label=f'Optimal Components ({optimal_n_components})')
        plt.xlabel('Number of Components')
        plt.ylabel('Cumulative Explained Variance')
        plt.title('Cumulative Explained Variance')
        plt.grid(True)
        plt.legend()
        
        plt.tight_layout()
        if save_path:
            plt.savefig(os.path.join(save_path, 'pca_variance_analysis.png'))
        plt.show()
    
    print(f"Optimal number of components for {variance_threshold:.0%} variance: {optimal_n_components}")
    print(f"Total components analyzed: {len(explained_variance_ratio)}")
    
    # 保存检查点
    result = (optimal_n_components, explained_variance_ratio, cumulative_variance)
    checkpoint_mgr.save_checkpoint('pca_variance', result, params)
    
    return result

In [ ]:
# 单元格 D: 修改后的最佳聚类分析函数 (单元格6)
def analyze_optimal_clusters(data, min_clusters=2, max_clusters=10, methods=['kmeans', 'spectral', 'agglomerative', 'gmm', 'dbscan'], save_path=None):
    """
    Analyze optimal number of clusters and clustering methods
    
    Parameters:
        data: input data
        min_clusters: minimum number of clusters
        max_clusters: maximum number of clusters
        methods: list of clustering methods to try
        save_path: path to save charts
        
    Returns:
        best_results: dictionary of best clustering results
    """
    import time
    
    # 检查是否有检查点
    params = {
        'data_shape': data.shape,
        'min_clusters': min_clusters,
        'max_clusters': max_clusters,
        'methods': methods
    }
    checkpoint_result = checkpoint_mgr.load_checkpoint('optimal_clusters', params)
    if checkpoint_result is not None:
        print("Loading cluster analysis from checkpoint...")
        
        # 重新绘制图表
        best_results = checkpoint_result
        
        # 提取保存的评分
        silhouette_scores = {method: [] for method in methods if method != 'dbscan'}
        calinski_scores = {method: [] for method in methods if method != 'dbscan'}
        davies_scores = {method: [] for method in methods if method != 'dbscan'}
        
        for method in methods:
            if method != 'dbscan':
                for n_clusters in range(min_clusters, max_clusters+1):
                    if n_clusters in best_results.get(method, {}).get('scores', {}):
                        scores = best_results[method]['scores'][n_clusters]
                        if 'silhouette' in scores:
                            silhouette_scores[method].append(scores['silhouette'])
                        if 'calinski' in scores:
                            calinski_scores[method].append(scores['calinski'])
                        if 'davies' in scores:
                            davies_scores[method].append(scores['davies'])
        
        # Plot evaluation metrics if there are results
        if any(len(scores) > 0 for scores in silhouette_scores.values()):
            plt.figure(figsize=(15, 12))
            
            # Silhouette Score (higher is better)
            plt.subplot(3, 1, 1)
            for method in methods:
                if method != 'dbscan' and len(silhouette_scores[method]) > 0:
                    plt.plot(range(min_clusters, min_clusters + len(silhouette_scores[method])),
                           silhouette_scores[method], 'o-', label=method)
            plt.title('Silhouette Score (Higher is Better)')
            plt.xlabel('Number of Clusters')
            plt.ylabel('Silhouette Score')
            plt.grid(True)
            plt.legend()

            # Calinski-Harabasz Index (higher is better)
            plt.subplot(3, 1, 2)
            for method in methods:
                if method != 'dbscan' and len(calinski_scores[method]) > 0:
                    plt.plot(range(min_clusters, min_clusters + len(calinski_scores[method])),
                           calinski_scores[method], 'o-', label=method)
            plt.title('Calinski-Harabasz Index (Higher is Better)')
            plt.xlabel('Number of Clusters')
            plt.ylabel('Calinski-Harabasz Index')
            plt.grid(True)
            plt.legend()
            
            # Davies-Bouldin Index (lower is better)
            plt.subplot(3, 1, 3)
            for method in methods:
                if method != 'dbscan' and len(davies_scores[method]) > 0:
                    plt.plot(range(min_clusters, min_clusters + len(davies_scores[method])),
                           davies_scores[method], 'o-', label=method)
            plt.title('Davies-Bouldin Index (Lower is Better)')
            plt.xlabel('Number of Clusters')
            plt.ylabel('Davies-Bouldin Index')
            plt.grid(True)
            plt.legend()

            plt.tight_layout()
            
            if save_path:
                plt.savefig(os.path.join(save_path, 'cluster_analysis.png'))
            plt.show()
        
        # Print best results
        print("\nBest clustering results:")
        for method, result in best_results.items():
            if method != 'dbscan' and 'n_clusters' in result:
                n_clusters = result['n_clusters']
                print(f"\n{method.upper()} best number of clusters: {n_clusters}")
                if 'silhouette' in result:
                    print(f"  Silhouette Score: {result['silhouette']:.4f}")
                    print(f"  Calinski-Harabasz Index: {result['calinski']:.1f}")
                    print(f"  Davies-Bouldin Index: {result['davies']:.4f}")
            elif method == 'dbscan' and 'n_clusters' in result:
                n_clusters = result['n_clusters']
                print(f"\n{method.upper()} number of clusters: {n_clusters}")
                if 'silhouette' in result:
                    print(f"  Silhouette Score: {result['silhouette']:.4f}")
                    print(f"  Calinski-Harabasz Index: {result['calinski']:.1f}")
                    print(f"  Davies-Bouldin Index: {result['davies']:.4f}")
            
            # Print distribution if available
            if 'labels' in result:
                labels = result['labels']
                unique, counts = np.unique(labels, return_counts=True)
                dist_str = ", ".join([f"Cluster{int(u)}:{c}" for u, c in zip(unique, counts)])
                print(f"  Cluster distribution: {dist_str}")
        
        return best_results
    
    # 如果没有检查点，执行聚类分析
    
    # If data is too large, subsample
    if data.shape[0] > 100000:
        from sklearn.model_selection import train_test_split
        _, data = train_test_split(data, test_size=100000/data.shape[0], random_state=RANDOM_SEED)
        print(f"Data too large, subsampled to {data.shape[0]} samples")
    
    # Initialize result storage
    silhouette_scores = {method: [] for method in methods if method != 'dbscan'}
    calinski_scores = {method: [] for method in methods if method != 'dbscan'}
    davies_scores = {method: [] for method in methods if method != 'dbscan'}
    cluster_labels = {method: {} for method in methods}
    
    # 创建结构存储所有分数，用于检查点
    all_scores = {method: {'scores': {}} for method in methods}
    
    print("Analyzing optimal number of clusters...")
    
    # Try different numbers of clusters and methods
    for n_clusters in range(min_clusters, max_clusters+1):
        print(f"\nTrying {n_clusters} clusters:")
        
        for method in methods:
            if method == 'dbscan' and n_clusters != min_clusters:
                # DBSCAN doesn't take n_clusters parameter, so only run once
                continue
                
            start_time = time.time()
            print(f"  Method: {method}...", end="", flush=True)
            
            # Create clustering model
            if method == 'kmeans':
                cluster_model = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=5)
            elif method == 'spectral':
                cluster_model = SpectralClustering(n_clusters=n_clusters, random_state=RANDOM_SEED, 
                                                 affinity='nearest_neighbors', n_neighbors=min(30, data.shape[0]//100))
            elif method == 'agglomerative':
                cluster_model = AgglomerativeClustering(n_clusters=n_clusters)
            elif method == 'gmm':
                cluster_model = GaussianMixture(n_components=n_clusters, random_state=RANDOM_SEED, n_init=5)
            elif method == 'dbscan':
                # For DBSCAN, we'll estimate eps based on data
                from sklearn.neighbors import NearestNeighbors
                nn = NearestNeighbors(n_neighbors=min(10, data.shape[0]//100))
                nn.fit(data)
                distances, _ = nn.kneighbors(data)
                avg_dist = np.mean(distances[:, 1:])  # Exclude self-distance
                
                cluster_model = DBSCAN(eps=avg_dist, min_samples=min(5, data.shape[0]//1000))
            else:
                raise ValueError(f"Unsupported clustering method: {method}")
            
            # Perform clustering
            if method == 'gmm':
                labels = cluster_model.fit_predict(data)
            else:
                labels = cluster_model.fit_predict(data)
            
            # Store labels
            cluster_labels[method][n_clusters] = labels
            
            # Initialize scores dictionary for this method and n_clusters
            all_scores[method]['scores'][n_clusters] = {}
            
            # Calculate clustering evaluation metrics
            try:
                if len(np.unique(labels)) > 1:  # Ensure at least two clusters
                    # To speed up, use data samples for metrics
                    if data.shape[0] > 10000:
                        sample_idx = np.random.choice(data.shape[0], 10000, replace=False)
                        sample_data = data[sample_idx]
                        sample_labels = labels[sample_idx]
                        sil_score = silhouette_score(sample_data, sample_labels)
                        cal_score = calinski_harabasz_score(sample_data, sample_labels)
                        dav_score = davies_bouldin_score(sample_data, sample_labels)
                    else:
                        sil_score = silhouette_score(data, labels)
                        cal_score = calinski_harabasz_score(data, labels)
                        dav_score = davies_bouldin_score(data, labels)
                    
                    # Store scores in comprehensive dictionary
                    all_scores[method]['scores'][n_clusters]['silhouette'] = sil_score
                    all_scores[method]['scores'][n_clusters]['calinski'] = cal_score
                    all_scores[method]['scores'][n_clusters]['davies'] = dav_score
                    
                    if method != 'dbscan':
                        silhouette_scores[method].append(sil_score)
                        calinski_scores[method].append(cal_score)
                        davies_scores[method].append(dav_score)
                    
                    elapsed = time.time() - start_time
                    print(f" Done! ({elapsed:.1f}s)")
                    print(f"    Silhouette Score: {sil_score:.4f}")
                    print(f"    Calinski-Harabasz Index: {cal_score:.1f}")
                    print(f"    Davies-Bouldin Index: {dav_score:.4f}")
                    
                    # Count samples in each cluster
                    unique, counts = np.unique(labels, return_counts=True)
                    dist_str = ", ".join([f"Cluster{int(u)}:{c}" for u, c in zip(unique, counts)])
                    print(f"    Cluster distribution: {dist_str}")
                else:
                    print(f" Warning: {method} produced single or empty clusters")
                    if method != 'dbscan':
                        silhouette_scores[method].append(-1)
                        calinski_scores[method].append(-1)
                        davies_scores[method].append(float('inf'))
                        # Store in comprehensive dictionary
                        all_scores[method]['scores'][n_clusters]['silhouette'] = -1
                        all_scores[method]['scores'][n_clusters]['calinski'] = -1
                        all_scores[method]['scores'][n_clusters]['davies'] = float('inf')
            except Exception as e:
                print(f" Error computing metrics: {e}")
                if method != 'dbscan':
                    silhouette_scores[method].append(-1)
                    calinski_scores[method].append(-1)
                    davies_scores[method].append(float('inf'))
                    # Store in comprehensive dictionary
                    all_scores[method]['scores'][n_clusters]['silhouette'] = -1
                    all_scores[method]['scores'][n_clusters]['calinski'] = -1
                    all_scores[method]['scores'][n_clusters]['davies'] = float('inf')
    
    # Plot evaluation metrics
    plt.figure(figsize=(15, 12))
    
    # Silhouette Score (higher is better)
    plt.subplot(3, 1, 1)
    for method in methods:
        if method != 'dbscan' and len(silhouette_scores[method]) > 0:
            plt.plot(range(min_clusters, min_clusters + len(silhouette_scores[method])),
                   silhouette_scores[method], 'o-', label=method)
    plt.title('Silhouette Score (Higher is Better)')
    plt.xlabel('Number of Clusters')
    plt.ylabel('Silhouette Score')
    plt.grid(True)
    plt.legend()

    # Calinski-Harabasz Index (higher is better)
    plt.subplot(3, 1, 2)
    for method in methods:
        if method != 'dbscan' and len(calinski_scores[method]) > 0:
            plt.plot(range(min_clusters, min_clusters + len(calinski_scores[method])),
                   calinski_scores[method], 'o-', label=method)
    plt.title('Calinski-Harabasz Index (Higher is Better)')
    plt.xlabel('Number of Clusters')
    plt.ylabel('Calinski-Harabasz Index')
    plt.grid(True)
    plt.legend()
    
    # Davies-Bouldin Index (lower is better)
    plt.subplot(3, 1, 3)
    for method in methods:
        if method != 'dbscan' and len(davies_scores[method]) > 0:
            plt.plot(range(min_clusters, min_clusters + len(davies_scores[method])),
                   davies_scores[method], 'o-', label=method)
    plt.title('Davies-Bouldin Index (Lower is Better)')
    plt.xlabel('Number of Clusters')
    plt.ylabel('Davies-Bouldin Index')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    
    if save_path:
        plt.savefig(os.path.join(save_path, 'cluster_analysis.png'))
    plt.show()
    
    # Find best results
    best_results = {}
    for method in methods:
        if method != 'dbscan' and len(silhouette_scores[method]) > 0:
            best_idx = np.argmax(silhouette_scores[method])
            best_n = min_clusters + best_idx
            best_results[method] = {
                'n_clusters': best_n,
                'silhouette': silhouette_scores[method][best_idx],
                'calinski': calinski_scores[method][best_idx],
                'davies': davies_scores[method][best_idx],
                'labels': cluster_labels[method][best_n]
            }
        elif method == 'dbscan':
            # For DBSCAN, just use the results from the one run
            labels = cluster_labels[method][min_clusters]
            n_clusters = len(np.unique(labels[labels >= 0]))  # Exclude noise points (-1)
            best_results[method] = {
                'n_clusters': n_clusters,
                'labels': labels
            }
            # Calculate metrics if possible
            if n_clusters > 1:
                if data.shape[0] > 10000:
                    sample_idx = np.random.choice(data.shape[0], 10000, replace=False)
                    sample_data = data[sample_idx]
                    sample_labels = labels[sample_idx]
                    best_results[method]['silhouette'] = silhouette_score(sample_data, sample_labels)
                    best_results[method]['calinski'] = calinski_harabasz_score(sample_data, sample_labels)
                    best_results[method]['davies'] = davies_bouldin_score(sample_data, sample_labels)
                else:
                    best_results[method]['silhouette'] = silhouette_score(data, labels)
                    best_results[method]['calinski'] = calinski_harabasz_score(data, labels)
                    best_results[method]['davies'] = davies_bouldin_score(data, labels)
    
    # Merge best results with all scores for comprehensive checkpoint
    for method in methods:
        if method in best_results:
            all_scores[method].update(best_results[method])
    
    # Print best results
    print("\nBest clustering results:")
    for method, result in best_results.items():
        n_clusters = result['n_clusters']
        print(f"\n{method.upper()} best number of clusters: {n_clusters}")
        if 'silhouette' in result:
            print(f"  Silhouette Score: {result['silhouette']:.4f}")
            print(f"  Calinski-Harabasz Index: {result['calinski']:.1f}")
            print(f"  Davies-Bouldin Index: {result['davies']:.4f}")
        
        # Count distribution
        labels = result['labels']
        unique, counts = np.unique(labels, return_counts=True)
        dist_str = ", ".join([f"Cluster{int(u)}:{c}" for u, c in zip(unique, counts)])
        print(f"  Cluster distribution: {dist_str}")
    
    # 保存检查点
    checkpoint_mgr.save_checkpoint('optimal_clusters', all_scores, params)
    
    return all_scores

In [ ]:
# 单元格 7: 可视化聚类结果与标签对比函数
def visualize_cluster_vs_labels(cluster_labels, original_labels, cluster_method, class_names=None, save_path=None):
    """
    Visualize the correspondence between clustering results and original labels
    
    Parameters:
        cluster_labels: clustering labels
        original_labels: original labels
        cluster_method: clustering method name
        class_names: list of class names
        save_path: path to save the chart
    """
    # Create confusion matrix
    # Rows are original classes, columns are clusters
    unique_clusters = np.unique(cluster_labels)
    unique_labels = np.unique(original_labels)
    n_clusters = len(unique_clusters)
    n_labels = len(unique_labels)
    
    matrix = np.zeros((n_labels, n_clusters))
    for i, label in enumerate(unique_labels):
        for j, cluster in enumerate(unique_clusters):
            # Count samples that belong to both this label and this cluster
            matrix[i, j] = np.sum((original_labels == label) & (cluster_labels == cluster))
    
    # Calculate row-normalized matrix (distribution of each original class)
    row_normalized = matrix.copy()
    row_sums = row_normalized.sum(axis=1, keepdims=True)
    row_normalized = np.divide(row_normalized, row_sums, where=row_sums!=0)
    
    # Plot heatmap
    plt.figure(figsize=(15, 10))
    
    # Use class names if provided
    if class_names is not None:
        y_labels = [class_names[l] if l < len(class_names) else f'Class {l}' for l in unique_labels]
    else:
        y_labels = [f'Class {l}' for l in unique_labels]
    
    # Plot original confusion matrix
    plt.subplot(1, 2, 1)
    sns.heatmap(matrix, annot=False, fmt='g', cmap='Blues',
            xticklabels=[f'Cluster {c}' for c in unique_clusters],
            yticklabels=y_labels)
    plt.title(f'{cluster_method} Clustering Results vs. Original Labels')
    plt.xlabel('Clustering Results')
    plt.ylabel('Original Labels')
    
    # Plot row-normalized confusion matrix
    plt.subplot(1, 2, 2)
    sns.heatmap(row_normalized, annot=False, fmt='.2f', cmap='Blues',
            xticklabels=[f'Cluster {c}' for c in unique_clusters],
            yticklabels=y_labels)
    plt.title(f'{cluster_method} Results vs. Original Labels (Normalized)')
    plt.xlabel('Clustering Results')
    plt.ylabel('Original Labels')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(os.path.join(save_path, f'{cluster_method}_cluster_vs_labels.png'))
    plt.show()
    
    # Calculate ARI score
    ari_score = adjusted_rand_score(original_labels, cluster_labels)
    print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
    
    # Print top clusters for each class
    print(f"\n{cluster_method} Clustering Analysis:")
    
    # Limit to top 5 classes to avoid excessive output
    top_classes = 5
    if len(unique_labels) > top_classes:
        # Select top classes by frequency
        class_counts = np.bincount(original_labels.astype(int))
        top_indices = np.argsort(class_counts)[::-1][:top_classes]
        print(f"Showing top {top_classes} classes by frequency:")
        selected_labels = top_indices
    else:
        selected_labels = unique_labels
    
    for i, label in enumerate(selected_labels):
        if label in unique_labels:
            label_idx = np.where(unique_labels == label)[0][0]
            # Get cluster distribution for this class
            class_dist = row_normalized[label_idx]
            # Find dominant cluster
            dominant_cluster = unique_clusters[np.argmax(class_dist)]
            dominant_percentage = np.max(class_dist) * 100
            
            class_name = y_labels[label_idx] if label_idx < len(y_labels) else f'Class {label}'
            print(f"Class {class_name}: Dominant Cluster = {dominant_cluster} ({dominant_percentage:.1f}%)")

In [ ]:
# 单元格 8: 特征组预处理函数（续）
def enhanced_preprocess_feature_groups(data, feature_indices, labels, normalize_method='robust', apply_feature_selection=True, n_features=None, save_path=None, pca_variance_threshold=0.95):
    """
    Enhanced feature group preprocessing
    
    Parameters:
        data: input data
        feature_indices: dictionary of feature indices
        labels: class labels
        normalize_method: normalization method, 'standard' or 'robust'
        apply_feature_selection: whether to apply feature selection
        n_features: dictionary of number of features to select for each group
        save_path: path to save results
        pca_variance_threshold: variance threshold for PCA (0.95 = 95%)
        
    Returns:
        processed_groups: processed feature groups
        selected_indices: selected feature indices
    """
    processed_groups = {}
    selected_indices = {}
    pca_models = {}
    
    # Set default feature counts
    if n_features is None:
        n_features = {
            'diffusion': min(10, len(feature_indices['diffusion'])),
            'qti': min(20, len(feature_indices['qti'])),
            'cest': min(15, len(feature_indices['cest']))
        }
    
    # Process each feature group
    for group_name, indices in feature_indices.items():
        print(f"\nProcessing {group_name} feature group...")
        
        # Extract features
        group_data = data[:, indices]
        
        # Normalization
        if normalize_method.lower() == 'standard':
            print(f"  Standardizing {group_name} features using StandardScaler...")
            scaler = StandardScaler()
            group_data = scaler.fit_transform(group_data)
        elif normalize_method.lower() == 'robust':
            print(f"  Normalizing {group_name} features using RobustScaler...")
            scaler = RobustScaler()
            group_data = scaler.fit_transform(group_data)
        
        # Feature selection
        if apply_feature_selection:
            k = min(n_features.get(group_name, group_data.shape[1] // 2), group_data.shape[1])
            print(f"  Selecting top {k} features from {group_name} feature group...")
            selected_data, indices, scores = select_discriminative_features(
                group_data, labels, group_name, k=k, save_path=save_path
            )
            processed_groups[group_name] = selected_data
            selected_indices[group_name] = indices
        else:
            processed_groups[group_name] = group_data
            
    # Print dimensions after processing
    for group_name, group_data in processed_groups.items():
        print(f"  {group_name} feature group dimension after processing: {group_data.shape}")
    
    return processed_groups, selected_indices

In [ ]:
# 单元格 9: 类别可分性分析函数
def analyze_class_separability(feature_groups, labels, save_path=None, use_sampling=True, sample_ratio=0.1):
    """
    Analyze class separability in feature space
    
    Parameters:
        feature_groups: processed feature groups
        labels: class labels
        save_path: path to save results
        use_sampling: whether to use sampling to speed up analysis
        sample_ratio: sampling ratio
        
    Returns:
        separability_scores: separability scores for each feature group
    """
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    from sklearn.model_selection import StratifiedShuffleSplit
    import time
    
    separability_scores = {}
    
    print("Analyzing class separability in feature space...")
    
    # If using sampling, create a sample subset
    if use_sampling:
        print(f"Using {sample_ratio*100:.1f}% of data for analysis")
        from sklearn.model_selection import train_test_split
        
        # Use stratified sampling to maintain class distribution
        indices = np.arange(len(labels))
        _, sampled_indices, _, sampled_labels = train_test_split(
            indices, labels, test_size=sample_ratio, stratify=labels, random_state=RANDOM_SEED
        )
        
        # Print sample counts and class distribution
        print(f"Original data: {len(labels)} samples")
        print(f"After sampling: {len(sampled_labels)} samples")
        unique, counts = np.unique(sampled_labels, return_counts=True)
        print(f"Sample class distribution: {len(unique)} unique classes")
        
        # Sample each feature group
        sampled_groups = {}
        for group_name, group_data in feature_groups.items():
            sampled_groups[group_name] = group_data[sampled_indices]
        
        # Use sampled data
        analysis_groups = sampled_groups
        analysis_labels = sampled_labels
    else:
        # Use all data
        analysis_groups = feature_groups
        analysis_labels = labels
    
    # Use cross-validation splitter instead of full k-fold CV
    cv_splitter = StratifiedShuffleSplit(n_splits=3, test_size=0.3, random_state=RANDOM_SEED)
    
    for group_name, group_data in analysis_groups.items():
        print(f"\nAnalyzing {group_name} feature group...")
        
        # Evaluate separability using multiple classifiers
        classifiers = {
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'SVM': SVC(kernel='rbf', C=1, probability=True),
            'RF': RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED)
        }
        
        group_scores = {}
        
        for clf_name, clf in classifiers.items():
            start_time = time.time()
            print(f"  Computing {clf_name} classifier performance...", end="", flush=True)
            
            # Use custom cross-validation instead of full cross_val_score
            scores = []
            for train_idx, test_idx in cv_splitter.split(group_data, analysis_labels):
                X_train, X_test = group_data[train_idx], group_data[test_idx]
                y_train, y_test = analysis_labels[train_idx], analysis_labels[test_idx]
                
                clf.fit(X_train, y_train)
                score = clf.score(X_test, y_test)
                scores.append(score)
            
            mean_score = np.mean(scores)
            std_score = np.std(scores)
            
            elapsed = time.time() - start_time
            print(f" Done! ({elapsed:.1f}s)")
            print(f"    Result: {mean_score:.4f} ± {std_score:.4f}")
            
            group_scores[clf_name] = {
                'mean': mean_score,
                'std': std_score
            }
        
        # Calculate average score as overall separability score
        mean_separability = np.mean([s['mean'] for s in group_scores.values()])
        separability_scores[group_name] = {
            'overall': mean_separability,
            'classifiers': group_scores
        }
        
        print(f"  Overall separability score: {mean_separability:.4f}")
    
    # Visualize results
    plt.figure(figsize=(12, 8))
    
    # Prepare data
    groups = list(separability_scores.keys())
    clf_names = list(separability_scores[groups[0]]['classifiers'].keys())
    
    x = np.arange(len(groups))
    width = 0.25
    offsets = np.linspace(-width, width, len(clf_names))
    
    # Plot bar chart
    for i, clf_name in enumerate(clf_names):
        means = [separability_scores[g]['classifiers'][clf_name]['mean'] for g in groups]
        stds = [separability_scores[g]['classifiers'][clf_name]['std'] for g in groups]
        
        plt.bar(x + offsets[i], means, width, label=clf_name, yerr=stds)
    
    # Plot random guessing baseline
    plt.axhline(y=1/len(np.unique(labels)), color='r', linestyle='--', 
           label=f'Random Guessing ({1/len(np.unique(labels)):.4f})')

    plt.xlabel('Feature Groups')
    plt.ylabel('Cross-Validation Accuracy')
    plt.title('Class Separability of Different Feature Groups')
    plt.xticks(x, groups)
    plt.legend()
    plt.grid(axis='y')

    
    if save_path:
        plt.savefig(os.path.join(save_path, 'class_separability.png'))
    plt.show()
    
    return separability_scores

In [ ]:
# 单元格 10: 计算类内和类间距离函数
def calculate_class_distances(data, labels):
    """
    Calculate within-class and between-class distances
    
    Parameters:
        data: input data
        labels: class labels
        
    Returns:
        within_class_distances: average within-class distances
        between_class_distances: average between-class distances
        fisher_ratio: Fisher's discriminant ratio
    """
    unique_labels = np.unique(labels)
    n_classes = len(unique_labels)
    
    # Compute class centroids
    centroids = np.zeros((n_classes, data.shape[1]))
    for i, label in enumerate(unique_labels):
        mask = labels == label
        centroids[i] = np.mean(data[mask], axis=0)
    
    # Compute within-class distances (average distance to centroid)
    within_class_distances = np.zeros(n_classes)
    for i, label in enumerate(unique_labels):
        mask = labels == label
        class_samples = data[mask]
        if len(class_samples) > 0:
            # Calculate squared Euclidean distance to centroid
            distances = np.sum((class_samples - centroids[i])**2, axis=1)
            within_class_distances[i] = np.mean(distances)
    
    # Compute between-class distances (pairwise distances between centroids)
    between_class_distances = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(i+1, n_classes):
            # Squared Euclidean distance between centroids
            dist = np.sum((centroids[i] - centroids[j])**2)
            between_class_distances[i, j] = dist
            between_class_distances[j, i] = dist
    
    # Average within-class and between-class distances
    avg_within = np.mean(within_class_distances)
    avg_between = np.mean(between_class_distances) / 2  # Divide by 2 to avoid double counting
    
    # Calculate Fisher's discriminant ratio
    # Higher values indicate better separability
    fisher_ratio = avg_between / (avg_within + 1e-10)  # Avoid division by zero
    
    return avg_within, avg_between, fisher_ratio

In [ ]:
# 广义判别值计算函数
def calculate_generalized_discrimination_value(data, labels):
    """
    Calculate Generalized Discrimination Value (GDV) according to the paper
    'How deep is deep enough? Quantifying class separability in the hidden layers of deep neural networks'
    
    Parameters:
        data: input data
        labels: class labels
        
    Returns:
        gdv: generalized discrimination value (more negative is better)
    """
    unique_labels = np.unique(labels)
    n_classes = len(unique_labels)
    n_samples, n_dims = data.shape
    
    if n_classes <= 1:
        return 0.0  # No discrimination possible with one class
    
    # Step 1: Z-score normalization + scaling to [-1, 1]
    normalized_data = np.zeros_like(data, dtype=np.float64)
    for d in range(n_dims):
        feature = data[:, d]
        mean = np.mean(feature)
        std = np.std(feature)
        if std > 0:  # Avoid division by zero
            normalized_data[:, d] = 0.5 * (feature - mean) / std
        else:
            normalized_data[:, d] = 0  # If std=0, set all values to 0
    
    # Step 2 & 3: Compute intra-class and inter-class distances
    intra_class_distances = np.zeros(n_classes)
    class_samples = {}
    
    # Precompute samples by class for faster access
    for l, label in enumerate(unique_labels):
        mask = labels == label
        class_samples[l] = normalized_data[mask]
    
    # Calculate average intra-class distances
    for l, label in enumerate(unique_labels):
        samples = class_samples[l]
        n_l = len(samples)
        
        if n_l <= 1:  # Skip if only one sample
            continue
        
        # For efficiency, compute pairwise distances using vectorized operations
        # Compute squared Euclidean distances between all pairs
        total_distance = 0
        count = 0
        
        # If class is too large, sample pairs to estimate average distance
        if n_l > 1000:
            # Sample 1000 random pairs
            pairs = 1000
            for _ in range(pairs):
                i, j = np.random.choice(n_l, 2, replace=False)
                total_distance += np.sqrt(np.sum((samples[i] - samples[j])**2))
            count = pairs
        else:
            # Compute all pairs for smaller classes
            for i in range(n_l-1):
                for j in range(i+1, n_l):
                    total_distance += np.sqrt(np.sum((samples[i] - samples[j])**2))
                    count += 1
        
        if count > 0:
            intra_class_distances[l] = total_distance / count
    
    # Calculate average inter-class distances
    inter_class_distances = np.zeros((n_classes, n_classes))
    for l in range(n_classes-1):
        samples_l = class_samples[l]
        n_l = len(samples_l)
        
        for m in range(l+1, n_classes):
            samples_m = class_samples[m]
            n_m = len(samples_m)
            
            if n_l == 0 or n_m == 0:  # Skip if either class is empty
                continue
            
            # For efficiency with large classes, sample pairs
            if n_l * n_m > 10000:
                # Sample 10000 random pairs
                total_distance = 0
                pairs = 10000
                for _ in range(pairs):
                    i = np.random.randint(n_l)
                    j = np.random.randint(n_m)
                    total_distance += np.sqrt(np.sum((samples_l[i] - samples_m[j])**2))
                avg_distance = total_distance / pairs
            else:
                # Compute all pairs for smaller classes
                distances = np.zeros((n_l, n_m))
                for i in range(n_l):
                    for j in range(n_m):
                        distances[i, j] = np.sqrt(np.sum((samples_l[i] - samples_m[j])**2))
                avg_distance = np.mean(distances)
            
            inter_class_distances[l, m] = avg_distance
            inter_class_distances[m, l] = avg_distance  # Symmetry
    
    # Step 4: Compute GDV
    # Average intra-class distance
    mean_intra_distance = np.mean(intra_class_distances)
    
    # Average inter-class distance
    sum_inter = 0
    count = 0
    for l in range(n_classes-1):
        for m in range(l+1, n_classes):
            sum_inter += inter_class_distances[l, m]
            count += 1
    
    mean_inter_distance = sum_inter / max(1, count)
    
    # Final GDV calculation (intra - inter) / sqrt(D)
    gdv = (mean_intra_distance - mean_inter_distance) / np.sqrt(n_dims)
    
    return gdv

In [ ]:
# 单元格 12: 计算Hopkins统计量函数
def calculate_hopkins_statistic(data, sample_size=None, iterations=10):
    """
    Calculate Hopkins statistic to measure the clustering tendency
    
    Parameters:
        data: input data
        sample_size: number of points to sample (defaults to 0.1 * data.shape[0])
        iterations: number of iterations to compute the average
        
    Returns:
        hopkins_stat: Hopkins statistic (0.5=random, closer to 1=clusterable)
    """
    from sklearn.neighbors import NearestNeighbors
    import random
    
    if sample_size is None:
        sample_size = int(0.1 * data.shape[0])
    
    # Use minimum sample size if data is too small
    sample_size = min(sample_size, int(0.5 * data.shape[0]))
    
    d = data.shape[1]
    
    # Run several iterations and take the average
    hopkins_stats = []
    
    for _ in range(iterations):
        # Randomly sample from the data
        sample_indices = random.sample(range(data.shape[0]), sample_size)
        sample_points = data[sample_indices]
        
        # Create a set of uniform random points within the data space
        min_vals = np.min(data, axis=0)
        max_vals = np.max(data, axis=0)
        uniform_points = np.random.uniform(min_vals, max_vals, (sample_size, d))
        
        # Find nearest neighbors for the sample points
        nbrs = NearestNeighbors(n_neighbors=2).fit(data)
        distances_sample, _ = nbrs.kneighbors(sample_points)
        # Use the distance to the 1st neighbor (not to self)
        u_distances = distances_sample[:, 1]
        
        # Find nearest neighbors for the uniform random points
        distances_uniform, _ = nbrs.kneighbors(uniform_points)
        w_distances = distances_uniform[:, 0]  # Distance to the nearest point
        
        # Calculate Hopkins statistic
        u_sum = np.sum(u_distances)
        w_sum = np.sum(w_distances)
        
        # H = w_sum / (u_sum + w_sum)
        # Avoid division by zero
        if u_sum + w_sum == 0:
            h = 0.5
        else:
            h = w_sum / (u_sum + w_sum)
        
        hopkins_stats.append(h)
    
    hopkins_stat = np.mean(hopkins_stats)
    return hopkins_stat

In [ ]:
# 单元格 13: 综合可分性分析函数
def comprehensive_separability_analysis(feature_groups, labels, save_path=None):
    """
    Perform comprehensive separability analysis on feature groups
    
    Parameters:
        feature_groups: dictionary of feature groups
        labels: class labels
        save_path: path to save results
        
    Returns:
        analysis_results: dictionary of analysis results
    """
    print("Performing comprehensive separability analysis...")
    
    analysis_results = {}
    
    # All metrics will be calculated for each feature group
    for group_name, group_data in feature_groups.items():
        print(f"\nAnalyzing {group_name} feature group...")
        
        # Initialize results for this group
        group_results = {}
        
        # 1. Calculate class distances
        print("  Calculating class distances...")
        avg_within, avg_between, fisher_ratio = calculate_class_distances(group_data, labels)
        group_results['within_class_distance'] = avg_within
        group_results['between_class_distance'] = avg_between
        group_results['fisher_ratio'] = fisher_ratio
        print(f"    Within-class distance: {avg_within:.4f}")
        print(f"    Between-class distance: {avg_between:.4f}")
        print(f"    Fisher ratio: {fisher_ratio:.4f}")
        
        # 2. Calculate Generalized Discrimination Value
        print("  Calculating Generalized Discrimination Value...")
        gdv = calculate_generalized_discrimination_value(group_data, labels)
        group_results['gdv'] = gdv
        print(f"    GDV: {gdv:.4f}")
        
        # 3. Calculate Hopkins statistic (clustering tendency)
        print("  Calculating Hopkins statistic...")
        hopkins_stat = calculate_hopkins_statistic(group_data)
        group_results['hopkins'] = hopkins_stat
        print(f"    Hopkins statistic: {hopkins_stat:.4f}")
        
        # Store results
        analysis_results[group_name] = group_results
    
    # Visualize comparative results
    plt.figure(figsize=(15, 10))
    
    # Plot Fisher ratio
    plt.subplot(2, 2, 1)
    groups = list(analysis_results.keys())
    fisher_values = [analysis_results[g]['fisher_ratio'] for g in groups]
    plt.bar(groups, fisher_values)
    plt.title('Fisher Discriminant Ratio')
    plt.ylabel('Ratio (higher is better)')
    plt.grid(axis='y')
    
    # Plot GDV
    plt.subplot(2, 2, 2)
    gdv_values = [analysis_results[g]['gdv'] for g in groups]
    plt.bar(groups, gdv_values)
    plt.title('Generalized Discrimination Value')
    plt.ylabel('GDV (higher is better)')
    plt.grid(axis='y')
    
    # Plot Hopkins statistic
    plt.subplot(2, 2, 3)
    hopkins_values = [analysis_results[g]['hopkins'] for g in groups]
    plt.bar(groups, hopkins_values)
    plt.axhline(y=0.5, color='r', linestyle='--', label='Random (0.5)')
    plt.title('Hopkins Statistic (Clustering Tendency)')
    plt.ylabel('Hopkins (closer to 1 = clusterable)')
    plt.legend()
    plt.grid(axis='y')
    
    # Plot distance ratio
    plt.subplot(2, 2, 4)
    within_values = [analysis_results[g]['within_class_distance'] for g in groups]
    between_values = [analysis_results[g]['between_class_distance'] for g in groups]
    x = np.arange(len(groups))
    width = 0.35
    plt.bar(x - width/2, within_values, width, label='Within-class')
    plt.bar(x + width/2, between_values, width, label='Between-class')
    plt.axhline(y=np.mean(within_values), color='r', linestyle='--', alpha=0.5)
    plt.axhline(y=np.mean(between_values), color='b', linestyle='--', alpha=0.5)
    plt.xticks(x, groups)
    plt.title('Within vs Between Class Distances')
    plt.ylabel('Average Distance')
    plt.legend()
    plt.grid(axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(os.path.join(save_path, 'comprehensive_separability.png'))
    plt.show()
    
    # Summary table
    print("\nSeparability analysis summary:")
    print(f"{'Group':<15}{'Fisher':<12}{'GDV':<12}{'Hopkins':<12}")
    print("-" * 50)
    for group in groups:
        print(f"{group:<15}{analysis_results[group]['fisher_ratio']:<12.4f}{analysis_results[group]['gdv']:<12.4f}{analysis_results[group]['hopkins']:<12.4f}")
    
    return analysis_results

In [ ]:
# 单元格 14: 数据加载函数
def load_multiclass_data_from_dirs(data_dirs, apply_pca=True, pca_variance_threshold=0.95, norm=True):
    """
    Load multiclass brain voxel data, using logic similar to original BrainVoxelSampler
    
    Parameters:
        data_dirs: dictionary containing training, testing, and validation data directories
        apply_pca: whether to apply PCA
        pca_variance_threshold: threshold for PCA variance retention
        norm: whether to normalize
    
    Returns:
        dictionary containing processed data
    """
    print(f"Loading multiclass data from directories...")
    print(f"Training directory: {data_dirs['train_dir']}")
    print(f"Testing directory: {data_dirs['test_dir']}")
    print(f"Validation directory: {data_dirs['val_dir']}")
    
    # Load data for all labels
    train_data_all = []
    train_labels_all = []
    test_data_all = []
    test_labels_all = []
    val_data_all = []
    val_labels_all = []
    
    # Get all voxel files in directories
    train_files = glob.glob(os.path.join(data_dirs['train_dir'], "label_*_count_*_voxels.npy"))
    test_files = glob.glob(os.path.join(data_dirs['test_dir'], "label_*_count_*_voxels.npy"))
    val_files = glob.glob(os.path.join(data_dirs['val_dir'], "label_*_count_*_voxels.npy"))
    
    print(f"Found {len(train_files)} training files, {len(test_files)} testing files, {len(val_files)} validation files")
    
    # Process training set
    for file_path in train_files:
        # Extract label ID from filename
        filename = os.path.basename(file_path)
        parts = filename.split('_')
        if len(parts) >= 4 and parts[0] == 'label':
            try:
                label_id = int(parts[1])
                # Ensure label is in valid range (0 to NUM_CLASS-1)
                if 0 <= label_id < NUM_CLASS:
                    print(f"Loading feature data: {filename}")
                    voxels = np.load(file_path)
                    labels = np.full(len(voxels), label_id)
                    train_data_all.append(voxels)
                    train_labels_all.append(labels)
                else:
                    print(f"Warning: Skipping label {label_id}, out of range [0, {NUM_CLASS-1}]")
            except ValueError:
                print(f"Warning: Could not extract label ID from {filename}")
    
    # Process test set
    for file_path in test_files:
        filename = os.path.basename(file_path)
        parts = filename.split('_')
        if len(parts) >= 4 and parts[0] == 'label':
            try:
                label_id = int(parts[1])
                if 0 <= label_id < NUM_CLASS:
                    print(f"Loading feature data: {filename}")
                    voxels = np.load(file_path)
                    labels = np.full(len(voxels), label_id)
                    test_data_all.append(voxels)
                    test_labels_all.append(labels)
                else:
                    print(f"Warning: Skipping label {label_id}, out of range [0, {NUM_CLASS-1}]")
            except ValueError:
                print(f"Warning: Could not extract label ID from {filename}")
    
    # Process validation set
    for file_path in val_files:
        filename = os.path.basename(file_path)
        parts = filename.split('_')
        if len(parts) >= 4 and parts[0] == 'label':
            try:
                label_id = int(parts[1])
                if 0 <= label_id < NUM_CLASS:
                    print(f"Loading feature data: {filename}")
                    voxels = np.load(file_path)
                    labels = np.full(len(voxels), label_id)
                    val_data_all.append(voxels)
                    val_labels_all.append(labels)
                else:
                    print(f"Warning: Skipping label {label_id}, out of range [0, {NUM_CLASS-1}]")
            except ValueError:
                print(f"Warning: Could not extract label ID from {filename}")
    
    # Combine all data
    if train_data_all:
        train_data = np.vstack(train_data_all)
        train_labels = np.concatenate(train_labels_all)
    else:
        raise ValueError("No valid training data found")
    
    if test_data_all:
        test_data = np.vstack(test_data_all)
        test_labels = np.concatenate(test_labels_all)
    else:
        raise ValueError("No valid testing data found")
    
    if val_data_all:
        val_data = np.vstack(val_data_all)
        val_labels = np.concatenate(val_labels_all)
    else:
        raise ValueError("No valid validation data found")
    
    # Print data ranges
    print(f"Training data range: {np.min(train_data)} to {np.max(train_data)}")
    print(f"Testing data range: {np.min(test_data)} to {np.max(test_data)}")
    print(f"Validation data range: {np.min(val_data)} to {np.max(val_data)}")
    
    # Check label ranges
    print(f"Training labels range: {np.min(train_labels)} to {np.max(train_labels)}")
    print(f"Testing labels range: {np.min(test_labels)} to {np.max(test_labels)}")
    print(f"Validation labels range: {np.min(val_labels)} to {np.max(val_labels)}")
    
    # Apply PCA
    if apply_pca:
        # Combine all data for PCA fitting
        all_data = np.vstack([train_data, test_data, val_data])
        
        # Determine PCA components based on variance threshold
        from sklearn.decomposition import PCA
        n_components, var_ratio, cum_var = analyze_pca_variance(all_data, variance_threshold=pca_variance_threshold, plot=True)
        pca_model = PCA(n_components=n_components)
        pca_model.fit(all_data)
        
        # Apply PCA transform
        train_data = pca_model.transform(train_data)
        test_data = pca_model.transform(test_data)
        val_data = pca_model.transform(val_data)
        
        # Normalize if needed
        if norm:
            # Compute normalization parameters based on all samples
            all_transformed = np.vstack([train_data, test_data, val_data])
            mins = np.min(all_transformed, axis=0)
            maxs = np.max(all_transformed, axis=0)
            ranges = maxs - mins + 1e-10  # Avoid division by zero
            
            # Apply normalization
            train_data = (train_data - mins) / ranges
            test_data = (test_data - mins) / ranges
            val_data = (val_data - mins) / ranges
        
        feature_dim = train_data.shape[1]
        print(f"Feature dimension after PCA: {feature_dim}")
    else:
        feature_dim = train_data.shape[1]
        pca_model = None
        print(f"PCA not applied, feature dimension: {feature_dim}")
    
    # Count samples per class
    class_counts = np.zeros((NUM_CLASS, 3), dtype=int)  # [train, test, val]
    for i in range(NUM_CLASS):
        class_counts[i, 0] = np.sum(train_labels == i)
        class_counts[i, 1] = np.sum(test_labels == i)
        class_counts[i, 2] = np.sum(val_labels == i)
    
    # Print class distribution
    print("\nClass distribution:")
    print(f"{'Class ID':^10}{'Train':^10}{'Test':^10}{'Val':^10}{'Total':^10}")
    print("-" * 50)
    for i in range(NUM_CLASS):
        if np.sum(class_counts[i]) > 0:  # Only print classes with samples
            total = np.sum(class_counts[i])
            print(f"{i:^10}{class_counts[i, 0]:^10}{class_counts[i, 1]:^10}{class_counts[i, 2]:^10}{total:^10}")
    
    total_train = len(train_labels)
    total_test = len(test_labels)
    total_val = len(val_labels)
    total_all = total_train + total_test + total_val
    print("-" * 50)
    print(f"{'Total':^10}{total_train:^10}{total_test:^10}{total_val:^10}{total_all:^10}")
    
    return {
        'train_samples': train_data,
        'train_labels': train_labels,
        'test_samples': test_data,
        'test_labels': test_labels,
        'val_samples': val_data,
        'val_labels': val_labels,
        'feature_dim': feature_dim,
        'pca_model': pca_model,
        'class_counts': class_counts
    }

In [ ]:
# 单元格 15: 大类定义函数
def define_big_classes():
    """
    Define big class label mappings based on neuroanatomy
    Map 102 fine-grained classes to 7 big classes

    Returns:
        fine_to_big: dictionary mapping fine classes to big classes
        big_to_fine: dictionary mapping big classes to fine classes
        big_class_names: list of big class names
    """
    # Define big class names
    big_class_names = [
        "Ventricular System",  # 0
        "White Matter",        # 1
        "Cortical Gray Matter", # 2
        "Deep Gray Nuclei",    # 3
        "Limbic System",       # 4
        "Brain Stem",          # 5
        "Other Structures"     # 6
    ]
    
    # Initialize mapping dictionaries
    fine_to_big = {}
    big_to_fine = {i: [] for i in range(len(big_class_names))}
    
    # 1. Ventricular System
    ventricular_system_classes = [1, 2, 9, 10, 14, 19, 34]
    for cls in ventricular_system_classes:
        fine_to_big[cls] = 0
        big_to_fine[0].append(cls)
    
    # 2. White Matter
    white_matter_classes = [
        3, 22, 96, 97, 98, 99, 100, 
        # Corpus callosum
        22, 23, 24, 25, 26,
        # White matter labels (wm-prefix)
        63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 
        81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 101, 102
    ]
    for cls in white_matter_classes:
        fine_to_big[cls] = 1
        if cls not in big_to_fine[1]:
            big_to_fine[1].append(cls)
    
    # 3. Cortical Gray Matter
    cortical_gray_matter_classes = [
        4, 23,  # Cerebellar cortex
        # Cerebral cortex labels (ctx-prefix)
        28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 
        46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62
    ]
    for cls in cortical_gray_matter_classes:
        fine_to_big[cls] = 2
        if cls not in big_to_fine[2]:
            big_to_fine[2].append(cls)
    
    # 4. Deep Gray Nuclei
    deep_gray_nuclei_classes = [
        5, 6, 7, 8, 15, 16, 17, 27,  # Thalamus, caudate, putamen, etc.
        24, 25, 26, 27, 30, 31, 32  # Right hemisphere structures
    ]
    for cls in deep_gray_nuclei_classes:
        fine_to_big[cls] = 3
        if cls not in big_to_fine[3]:
            big_to_fine[3].append(cls)
    
    # 5. Limbic System
    limbic_system_classes = [12, 13, 28, 29]  # Hippocampus, amygdala
    for cls in limbic_system_classes:
        fine_to_big[cls] = 4
        if cls not in big_to_fine[4]:
            big_to_fine[4].append(cls)
    
    # 6. Brain Stem
    brain_stem_classes = [11]
    for cls in brain_stem_classes:
        fine_to_big[cls] = 5
        big_to_fine[5].append(cls)
    
    # 7. Other Structures
    # Include all other structures not in the above categories
    for cls in range(NUM_CLASS):  # Assuming NUM_CLASS is defined as 102
        if cls not in fine_to_big:
            fine_to_big[cls] = 6
            big_to_fine[6].append(cls)
    
    # Print big class statistics
    print("Big Class Statistics:")
    for i, name in enumerate(big_class_names):
        print(f"Big Class {i} - {name}: {len(big_to_fine[i])} fine classes")
        print(f"    Fine classes included: {big_to_fine[i]}")
    
    return fine_to_big, big_to_fine, big_class_names


def map_to_big_classes(fine_labels, fine_to_big):
    """
    Map fine class labels to big class labels
    
    Parameters:
        fine_labels (array): fine class labels
        fine_to_big (dict): mapping from fine to big classes
    
    Returns:
        big_labels: big class labels
    """
    # Use numpy's vectorize function for efficiency
    mapper = np.vectorize(lambda x: fine_to_big.get(x, 6))  # Default map to "Other Structures" class
    big_labels = mapper(fine_labels)
    
    # Print big class distribution
    unique_classes, counts = np.unique(big_labels, return_counts=True)
    print("\nBig Class Label Distribution:")
    for cls, count in zip(unique_classes, counts):
        print(f"Big Class {cls}: {count} samples")
    
    return big_labels

In [ ]:
# 单元格 16: 可视化混淆矩阵函数
def plot_confusion_heatmap(true_labels, pred_labels, class_names=None, normalize=True, save_path=None, title="Confusion Matrix"):
    """
    Plot confusion matrix heatmap
    
    Parameters:
        true_labels: true labels
        pred_labels: predicted labels
        class_names: list of class names
        normalize: whether to normalize
        save_path: path to save plot
        title: plot title
    """
    # Compute confusion matrix
    cm = confusion_matrix(true_labels, pred_labels)
    
    # Normalize if requested
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm = np.nan_to_num(cm)  # Replace NaN with zero
    
    # Create figure
    plt.figure(figsize=(12, 10))
    
    # Create class labels for x and y axes
    if class_names is None:
        unique_labels = np.unique(np.concatenate([true_labels, pred_labels]))
        class_names = [f'Class {label}' for label in unique_labels]
    
    # Plot heatmap
    sns.heatmap(cm, annot=False, fmt='.2f' if normalize else 'g',
                cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    plt.show()
    
    # Print accuracy
    accuracy = np.sum(np.diag(cm)) / np.sum(cm)
    print(f"Accuracy: {accuracy:.4f}")
    
    # Print per-class accuracy
    class_accuracy = np.diag(cm) / np.sum(cm, axis=1)
    print("\nPer-class Accuracy:")
    for i, acc in enumerate(class_accuracy):
        class_name = class_names[i] if i < len(class_names) else f'Class {i}'
        print(f"{class_name}: {acc:.4f}")
    
    # Find most confused pairs
    np.fill_diagonal(cm, 0)  # Zero out diagonal to find off-diagonal confusion
    n_pairs = min(5, np.sum(cm > 0))  # Find top 5 or fewer if there are fewer confusions
    
    print("\nMost confused class pairs:")
    for _ in range(n_pairs):
        max_idx = np.unravel_index(np.argmax(cm), cm.shape)
        true_idx, pred_idx = max_idx
        conf_value = cm[true_idx, pred_idx]
        if conf_value > 0:
            true_cls = class_names[true_idx] if true_idx < len(class_names) else f'Class {true_idx}'
            pred_cls = class_names[pred_idx] if pred_idx < len(class_names) else f'Class {pred_idx}'
            print(f"True: {true_cls}, Predicted: {pred_cls}, Confusion: {conf_value:.4f}")
            cm[true_idx, pred_idx] = 0  # Zero out this entry to find the next highest
        else:
            break

In [ ]:
# 单元格 17: 比较聚类与大类的一致性函数
def compare_clustering_with_big_classes(cluster_labels, big_class_labels, cluster_method, big_class_names=None, save_path=None):
    """
    Compare consistency between clustering results and existing big class division
    
    Parameters:
        cluster_labels: clustering labels
        big_class_labels: big class labels
        cluster_method: clustering method name
        big_class_names: list of big class names
        save_path: path to save plot
    
    Returns:
        consistency_score: consistency score
    """
    # Create confusion matrix
    unique_clusters = np.unique(cluster_labels)
    unique_big_classes = np.unique(big_class_labels)
    n_clusters = len(unique_clusters)
    n_big_classes = len(unique_big_classes)
    
    matrix = np.zeros((n_big_classes, n_clusters))
    for i, big_class in enumerate(unique_big_classes):
        for j, cluster in enumerate(unique_clusters):
            # Count samples that belong to both this big class and this cluster
            matrix[i, j] = np.sum((big_class_labels == big_class) & (cluster_labels == cluster))
    
    # Calculate row-normalized matrix (distribution of each big class)
    row_normalized = matrix.copy()
    row_sums = row_normalized.sum(axis=1, keepdims=True)
    row_normalized = np.divide(row_normalized, row_sums, where=row_sums!=0)
    
    # Calculate column-normalized matrix (distribution of each cluster)
    col_normalized = matrix.copy()
    col_sums = col_normalized.sum(axis=0, keepdims=True)
    col_normalized = np.divide(col_normalized, col_sums, where=col_sums!=0)
    
    # Plot heatmaps
    plt.figure(figsize=(15, 12))
    
    # Use big class names if provided
    if big_class_names is not None:
        y_labels = big_class_names
    else:
        y_labels = [f'Big Class {l}' for l in unique_big_classes]
    
    # Plot original confusion matrix
    plt.subplot(2, 2, 1)
    sns.heatmap(matrix, annot=True, fmt='g', cmap='Blues',
            xticklabels=[f'Cluster {c}' for c in unique_clusters],
            yticklabels=y_labels)
    plt.title(f'{cluster_method} Clustering Results vs. Major Categories')
    plt.xlabel('Clustering Results')
    plt.ylabel('Major Categories')

    # Plot row-normalized confusion matrix
    plt.subplot(2, 2, 2)
    sns.heatmap(row_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'Cluster {c}' for c in unique_clusters],
            yticklabels=y_labels)
    plt.title('Row-Normalized - Cluster Distribution per Major Category')
    plt.xlabel('Clustering Results')
    plt.ylabel('Major Categories')

    # Plot column-normalized confusion matrix
    plt.subplot(2, 2, 3)
    sns.heatmap(col_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'Cluster {c}' for c in unique_clusters],
            yticklabels=y_labels)
    plt.title('Column-Normalized - Major Category Distribution per Cluster')
    plt.xlabel('Clustering Results')
    plt.ylabel('Major Categories')

    
    # Calculate consistency score
    # Use Hungarian algorithm to find best matching
    from scipy.optimize import linear_sum_assignment
    
    # Create cost matrix (to maximize consistency, so negate)
    cost_matrix = -matrix.copy()
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Calculate consistency score of best matching
    matched_samples = sum(matrix[row_ind[i], col_ind[i]] for i in range(len(row_ind)))
    total_samples = matrix.sum()
    consistency_score = matched_samples / total_samples
    
    # Display best matching and consistency score
    plt.subplot(2, 2, 4)
    plt.axis('off')
    plt.text(0.5, 0.9, 'Best Cluster-Major Category Match', ha='center', fontsize=14, fontweight='bold')
    plt.text(0.5, 0.8, f'Consistency Score: {consistency_score:.4f}', ha='center', fontsize=12)

    
    for i, (r, c) in enumerate(zip(row_ind, col_ind)):
        if i < 10:  # Only show first 10 matches to avoid overcrowding
            big_class_name = y_labels[r]
            cluster_name = f'Cluster {unique_clusters[c]}'
            match_score = matrix[r, c] / row_sums[r]
            plt.text(0.5, 0.7 - i*0.05, f'{big_class_name} ↔ {cluster_name} ({match_score[0]:.2f})', 
                    ha='center', fontsize=10)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(os.path.join(save_path, f'{cluster_method}_vs_big_classes.png'))
    plt.show()
    
    # Print dominant big class for each cluster
    print(f"\n{cluster_method} clustering vs. big classes correspondence:")
    for j, cluster in enumerate(unique_clusters):
        # Get counts of each big class in this cluster
        class_counts = matrix[:, j]
        # Find dominant big class
        dominant_idx = np.argmax(class_counts)
        dominant_percentage = np.max(class_counts) / np.sum(class_counts) * 100
        dominant_name = y_labels[dominant_idx]
        
        print(f"Cluster {cluster}: Dominant big class = {dominant_name} ({dominant_percentage:.1f}%)")
        
        # List top 3 dominant big classes
        top_indices = np.argsort(class_counts)[::-1][:3]
        for idx in top_indices:
            if class_counts[idx] > 0:
                big_name = y_labels[idx]
                print(f"  {big_name}: {class_counts[idx]} samples ({class_counts[idx]/np.sum(class_counts)*100:.1f}%)")
    
    return consistency_score

In [ ]:
# 单元格 18: 评估特征重要性函数
def evaluate_feature_importance(feature_groups, labels, method='random_forest', save_path=None):
    """
    Evaluate feature importance for each feature group
    
    Parameters:
        feature_groups: dictionary of feature groups
        labels: class labels
        method: method to evaluate feature importance ('random_forest' or 'f_score')
        save_path: path to save results
        
    Returns:
        importance_dict: dictionary of feature importance by group
    """
    importance_dict = {}
    
    print("Evaluating feature importance...")
    
    for group_name, group_data in feature_groups.items():
        print(f"\nAnalyzing {group_name} feature importance...")
        
        if method == 'random_forest':
            # Use Random Forest to evaluate feature importance
            rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
            rf.fit(group_data, labels)
            importance = rf.feature_importances_
            
            # Store results
            importance_dict[group_name] = importance
            
            # Plot feature importance
            plt.figure(figsize=(12, 6))
            
            # Sort features by importance
            indices = np.argsort(importance)[::-1]
            
            plt.bar(range(len(importance)), importance[indices])
            plt.title(f'{group_name} Feature Importance (Random Forest)')
            plt.xlabel('Feature Rank')
            plt.ylabel('Importance Score')
            plt.grid(axis='y')
            
            # Highlight top 10 features
            top_n = min(10, len(importance))
            top_indices = indices[:top_n]
            top_importances = importance[top_indices]
            
            print(f"Top {top_n} features by importance:")
            for i, idx in enumerate(top_indices):
                print(f"  Feature {idx}: Importance = {importance[idx]:.4f}")
            
            # Output feature importance stats
            print(f"  Mean importance: {np.mean(importance):.4f}")
            print(f"  Median importance: {np.median(importance):.4f}")
            print(f"  Top 10% features account for {np.sum(importance[indices[:int(0.1*len(importance))]])/np.sum(importance)*100:.1f}% of total importance")
            
            if save_path:
                plt.savefig(os.path.join(save_path, f'{group_name}_feature_importance.png'))
            plt.show()
            
        elif method == 'f_score':
            # Use F-score for feature importance
            selector = SelectKBest(f_classif, k='all')
            selector.fit(group_data, labels)
            importance = selector.scores_
            
            # Store results
            importance_dict[group_name] = importance
            
            # Plot feature importance
            plt.figure(figsize=(12, 6))
            
            # Sort features by importance
            indices = np.argsort(importance)[::-1]
            
            plt.bar(range(len(importance)), importance[indices])
            plt.title(f'{group_name} Feature Importance (F-score)')
            plt.xlabel('Feature Rank')
            plt.ylabel('F-score')
            plt.yscale('log')  # Log scale for F-scores
            plt.grid(axis='y')
            
            # Highlight top 10 features
            top_n = min(10, len(importance))
            top_indices = indices[:top_n]
            top_importances = importance[top_indices]
            
            print(f"Top {top_n} features by F-score:")
            for i, idx in enumerate(top_indices):
                print(f"  Feature {idx}: F-score = {importance[idx]:.4f}")
            
            if save_path:
                plt.savefig(os.path.join(save_path, f'{group_name}_f_score_importance.png'))
            plt.show()
    
    return importance_dict

In [ ]:
# 单元格 E: 修改后的run_complete_separability_analysis函数 (单元格19)
def run_complete_separability_analysis(data, labels, pca_enabled=True, save_path=None):
    """
    Run comprehensive separability analysis on the dataset with checkpoint support
    
    Parameters:
        data: input data
        labels: class labels
        pca_enabled: whether to use PCA (True) or raw features (False)
        save_path: path to save results
        
    Returns:
        analysis_results: dictionary of analysis results
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Create specific save path for this run
    if save_path:
        run_save_path = os.path.join(save_path, f"analysis_{'with_pca' if pca_enabled else 'raw'}_{timestamp}")
        if not os.path.exists(run_save_path):
            os.makedirs(run_save_path)
    else:
        run_save_path = None
    
    # 检查是否存在完整的分析结果检查点
    analysis_params = {
        'data_shape': data.shape,
        'pca_enabled': pca_enabled
    }
    complete_results = checkpoint_mgr.load_checkpoint('complete_analysis', analysis_params)
    if complete_results is not None:
        print(f"Loading complete analysis results from checkpoint...")
        return complete_results
    
    print(f"\n{'='*80}")
    print(f"RUNNING SEPARABILITY ANALYSIS {'WITH PCA' if pca_enabled else 'ON RAW FEATURES'}")
    print(f"{'='*80}")
    
    # Step 1: Define feature groups
    feature_indices = {
        'diffusion': DIFF_FEATURES,
        'qti': QTI_FEATURES,
        'cest': CEST_FEATURES
    }
    
    # Step 2: Apply PCA if enabled
    # 检查是否存在PCA检查点
    pca_params = {
        'data_shape': data.shape,
        'pca_enabled': pca_enabled
    }
    pca_results = checkpoint_mgr.load_checkpoint('pca_processing', pca_params)
    
    if pca_results is not None:
        print("\n1. LOADING PCA RESULTS FROM CHECKPOINT")
        processed_groups = pca_results
    else:
        if pca_enabled:
            print("\n1. APPLYING PCA")
            all_data_pca = data.copy()
            
            # Apply PCA to each feature group
            pca_data = {}
            pca_models = {}
            
            for group_name, indices in feature_indices.items():
                print(f"\nApplying PCA to {group_name} feature group...")
                group_data = data[:, indices]
                
                # Analyze optimal PCA components
                n_components, _, _ = analyze_pca_variance(
                    group_data, 
                    variance_threshold=PCA_VARIANCE_RATIO,
                    plot=True,
                    save_path=run_save_path
                )
                
                # Apply PCA
                pca = PCA(n_components=n_components)
                transformed_data = pca.fit_transform(group_data)
                
                pca_data[group_name] = transformed_data
                pca_models[group_name] = pca
                
                print(f"  Original dimensions: {group_data.shape}")
                print(f"  After PCA dimensions: {transformed_data.shape}")
            
            # Process with PCA-transformed features
            processed_groups, selected_indices = enhanced_preprocess_feature_groups(
                data, feature_indices, labels, 
                normalize_method='robust', apply_feature_selection=False,
                save_path=run_save_path
            )
            
            # Override processed groups with PCA data
            for group_name, pca_group_data in pca_data.items():
                processed_groups[group_name] = pca_group_data
        else:
            print("\n1. USING RAW FEATURES")
            # Process with raw features
            processed_groups, selected_indices = enhanced_preprocess_feature_groups(
                data, feature_indices, labels, 
                normalize_method='robust', apply_feature_selection=True,
                n_features={'diffusion': 15, 'qti': 30, 'cest': 20},
                save_path=run_save_path
            )
        
        # 保存PCA或原始特征处理结果
        checkpoint_mgr.save_checkpoint('pca_processing', processed_groups, pca_params)
    
    # Step 3: Dimensionality reduction visualization
    # 检查是否存在可视化检查点
    viz_params = {
        'groups': list(processed_groups.keys()),
        'data_shape': data.shape
    }
    viz_results = checkpoint_mgr.load_checkpoint('feature_visualization', viz_params)
    
    if viz_results is None:
        print("\n2. VISUALIZATION OF FEATURE SPACES")
        viz_methods = ['tsne', 'umap']
        viz_results = {}
        
        for group_name, group_data in processed_groups.items():
            print(f"\nVisualizing {group_name} feature space...")
            group_viz = {}
            
            for method in viz_methods:
                print(f"  Using {method.upper()}...")
                reduced_data, _ = advanced_feature_reduction(
                    group_data, method=method, n_components=2, labels=labels,
                    plot=True, title=f"{group_name} Features", 
                    save_path=os.path.join(run_save_path, f"{group_name}_{method}_visualization.png") if run_save_path else None
                )
                group_viz[method] = reduced_data
            
            viz_results[group_name] = group_viz
        
        # 保存可视化结果
        checkpoint_mgr.save_checkpoint('feature_visualization', viz_results, viz_params)
    else:
        print("\n2. LOADING FEATURE VISUALIZATION FROM CHECKPOINT")
    
    # Step 4: Comprehensive separability analysis
    # 检查是否存在可分性分析检查点
    sep_params = {
        'groups': list(processed_groups.keys()),
        'data_shape': data.shape
    }
    separability_results = checkpoint_mgr.load_checkpoint('comprehensive_separability', sep_params)
    
    if separability_results is None:
        print("\n3. COMPREHENSIVE SEPARABILITY METRICS")
        separability_results = comprehensive_separability_analysis(
            processed_groups, labels, save_path=run_save_path
        )
        
        # 保存可分性分析结果
        checkpoint_mgr.save_checkpoint('comprehensive_separability', separability_results, sep_params)
    else:
        print("\n3. LOADING COMPREHENSIVE SEPARABILITY METRICS FROM CHECKPOINT")
    
    # Step 5: Class separability using classifiers
    # 检查是否存在分类器可分性分析检查点
    clf_params = {
        'groups': list(processed_groups.keys()),
        'data_shape': data.shape
    }
    classifier_results = checkpoint_mgr.load_checkpoint('classifier_separability', clf_params)
    
    if classifier_results is None:
        print("\n4. CLASSIFIER-BASED SEPARABILITY ANALYSIS")
        classifier_results = analyze_class_separability(
            processed_groups, labels, save_path=run_save_path,
            use_sampling=False  # Use all data
        )
        
        # 保存分类器可分性分析结果
        checkpoint_mgr.save_checkpoint('classifier_separability', classifier_results, clf_params)
    else:
        print("\n4. LOADING CLASSIFIER SEPARABILITY RESULTS FROM CHECKPOINT")
    
    # Step 6: Clustering analysis
    # 检查是否存在聚类分析检查点
    clust_params = {
        'groups': list(processed_groups.keys()),
        'data_shape': data.shape
    }
    clustering_results = checkpoint_mgr.load_checkpoint('clustering_analysis', clust_params)
    
    if clustering_results is None:
        print("\n5. CLUSTERING ANALYSIS")
        clustering_results = {}
        for group_name, group_data in processed_groups.items():
            print(f"\nAnalyzing optimal clustering for {group_name}...")
            group_results = analyze_optimal_clusters(
                group_data, min_clusters=2, max_clusters=10,  
                methods=['kmeans', 'spectral', 'agglomerative', 'gmm'],
                save_path=run_save_path
            )
            clustering_results[group_name] = group_results
        
        # 保存聚类分析结果
        checkpoint_mgr.save_checkpoint('clustering_analysis', clustering_results, clust_params)
    else:
        print("\n5. LOADING CLUSTERING ANALYSIS FROM CHECKPOINT")
    
    # Step 7: Feature importance analysis
    # 检查是否存在特征重要性分析检查点
    imp_params = {
        'groups': list(processed_groups.keys()),
        'data_shape': data.shape
    }
    importance_results = checkpoint_mgr.load_checkpoint('feature_importance', imp_params)
    
    if importance_results is None:
        print("\n6. FEATURE IMPORTANCE ANALYSIS")
        importance_results = evaluate_feature_importance(
            processed_groups, labels, method='random_forest', save_path=run_save_path
        )
        
        # 保存特征重要性分析结果
        checkpoint_mgr.save_checkpoint('feature_importance', importance_results, imp_params)
    else:
        print("\n6. LOADING FEATURE IMPORTANCE ANALYSIS FROM CHECKPOINT")
    
    # Step 8: Big class mapping and analysis
    # 检查是否存在大类分析检查点
    big_class_params = {
        'data_shape': data.shape
    }
    big_class_results = checkpoint_mgr.load_checkpoint('big_class_analysis', big_class_params)
    
    if big_class_results is None:
        print("\n7. BIG CLASS ANALYSIS")
        fine_to_big, big_to_fine, big_class_names = define_big_classes()
        big_class_labels = map_to_big_classes(labels, fine_to_big)
        
        # Visualization with big classes
        print("\nVisualizing feature spaces with big class labels...")
        big_class_viz = {}
        for group_name, group_data in processed_groups.items():
            group_viz = {}
            for method in ['tsne', 'umap']:
                print(f"  {group_name} with {method.upper()} by big classes...")
                reduced_data, _ = advanced_feature_reduction(
                    group_data, method=method, n_components=2, labels=big_class_labels,
                    plot=True, title=f"{group_name} Features (Big Classes)", 
                    save_path=os.path.join(run_save_path, f"{group_name}_{method}_big_classes.png") if run_save_path else None
                )
                group_viz[method] = reduced_data
            big_class_viz[group_name] = group_viz
        
        # Cluster vs big class analysis
        print("\nAnalyzing clustering vs big classes...")
        consistency_scores = {}
        for group_name, group_results in clustering_results.items():
            best_method = max(group_results.items(), key=lambda x: x[1].get('silhouette', -1) if x[0] != 'dbscan' else -1)[0]
            best_labels = group_results[best_method]['labels']
            
            print(f"\nComparing {group_name} {best_method} clustering with big classes...")
            consistency = compare_clustering_with_big_classes(
                best_labels, big_class_labels, 
                f"{group_name}_{best_method}",
                big_class_names=big_class_names,
                save_path=run_save_path
            )
            consistency_scores[f"{group_name}_{best_method}"] = consistency
        
        # Fisher discriminant analysis for big classes
        print("\n8. FISHER DISCRIMINANT ANALYSIS FOR BIG CLASSES")
        big_class_separability = {}
        for group_name, group_data in processed_groups.items():
            print(f"\nCalculating Fisher ratios for {group_name} with big classes...")
            within, between, fisher = calculate_class_distances(group_data, big_class_labels)
            big_class_separability[group_name] = {
                'within': within,
                'between': between,
                'fisher': fisher
            }
            print(f"  Within-class distance: {within:.4f}")
            print(f"  Between-class distance: {between:.4f}")
            print(f"  Fisher ratio: {fisher:.4f}")
        
        # Visualize big class Fisher ratios
        plt.figure(figsize=(10, 6))
        groups = list(big_class_separability.keys())
        fisher_values = [big_class_separability[g]['fisher'] for g in groups]
        plt.bar(groups, fisher_values)
        plt.title('Fisher Discriminant Ratio (Big Classes)')
        plt.ylabel('Ratio (higher is better)')
        plt.grid(axis='y')
        if run_save_path:
            plt.savefig(os.path.join(run_save_path, 'big_class_fisher_ratios.png'))
        plt.show()
        
        # 保存大类分析结果
        big_class_results = {
            'fine_to_big': fine_to_big,
            'big_to_fine': big_to_fine,
            'big_class_names': big_class_names,
            'big_class_labels': big_class_labels,
            'big_class_viz': big_class_viz,
            'consistency_scores': consistency_scores,
            'big_class_separability': big_class_separability
        }
        checkpoint_mgr.save_checkpoint('big_class_analysis', big_class_results, big_class_params)
    else:
        print("\n7. LOADING BIG CLASS ANALYSIS FROM CHECKPOINT")
        fine_to_big = big_class_results['fine_to_big']
        big_to_fine = big_class_results['big_to_fine']
        big_class_names = big_class_results['big_class_names']
        big_class_labels = big_class_results['big_class_labels']
        consistency_scores = big_class_results['consistency_scores']
        big_class_separability = big_class_results['big_class_separability']
    
    # 最终结果汇总
    print("\n9. FINAL SUMMARY")
    print(f"\nOverall separability assessment ({'with PCA' if pca_enabled else 'raw features'}):")
    
    # Summarize separability by feature group
    print("\nFeature Group Separability:")
    for group_name in processed_groups.keys():
        fisher_fine = separability_results[group_name]['fisher_ratio']
        fisher_big = big_class_separability[group_name]['fisher']
        gdv = separability_results[group_name]['gdv']
        hopkins = separability_results[group_name]['hopkins']
        clf_acc = classifier_results[group_name]['overall']
        
        print(f"\n{group_name} feature group:")
        print(f"  Fisher ratio (fine classes): {fisher_fine:.4f}")
        print(f"  Fisher ratio (big classes): {fisher_big:.4f}")
        print(f"  Generalized discrimination value: {gdv:.4f}")
        print(f"  Classifier accuracy: {clf_acc:.4f}")
        print(f"  Hopkins statistic: {hopkins:.4f}")
    
    # Summarize clustering consistency
    print("\nClustering vs. Big Classes Consistency:")
    for key, consistency in consistency_scores.items():
        print(f"  {key}: {consistency:.4f}")
    
    # 汇总所有分析结果
    comprehensive_results = {
        'feature_groups': processed_groups,
        'separability': separability_results,
        'classifier_performance': classifier_results,
        'clustering': clustering_results,
        'big_class_separability': big_class_separability,
        'consistency_scores': consistency_scores,
        'feature_importance': importance_results,
        'pca_enabled': pca_enabled
    }
    
    # 保存完整分析结果
    checkpoint_mgr.save_checkpoint('complete_analysis', comprehensive_results, analysis_params)
    
    return comprehensive_results

In [ ]:
# # 单元格 F: 修改后的主执行单元格 (单元格20)
# # 设置data_dirs根据你的数据路径
# DATA_DIRS = {
#     'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
#     'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
#     'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"
# }

# # 创建可分析路径
# os.makedirs(SAVE_PATH, exist_ok=True)

# # 分析进度管理
# def show_analysis_progress():
#     """显示当前分析进度"""
#     progress = checkpoint_mgr.get_progress()
#     if not progress:
#         print("No analysis has been started yet.")
#         return
    
#     print("\nCurrent Analysis Progress:")
#     print("-" * 50)
#     for step, info in progress.items():
#         status = "✓ Completed" if info.get('completed', False) else "⚠ Incomplete"
#         timestamp = info.get('timestamp', 'N/A')
#         print(f"{step:30s}: {status} ({timestamp})")
#     print("-" * 50)

# # 是否从头开始分析
# def reset_analysis():
#     """重置所有分析进度"""
#     confirmation = input("Are you sure you want to reset all analysis progress? (y/n): ")
#     if confirmation.lower() == 'y':
#         checkpoint_mgr.reset_progress()
#         print("All analysis progress has been reset.")
#     else:
#         print("Reset cancelled.")

# # 显示当前进度
# show_analysis_progress()

# # 询问是否重置
# reset_option = input("Do you want to reset all analysis and start fresh? (y/n): ")
# if reset_option.lower() == 'y':
#     reset_analysis()

# # 数据加载部分，使用检查点管理
# data_params = {'data_dirs': DATA_DIRS}
# cached_data = checkpoint_mgr.load_checkpoint('data_loading', data_params)

# if cached_data is not None:
#     print("Loading dataset from checkpoint...")
#     val_data = cached_data['val_data']
#     val_labels = cached_data['val_labels']
#     print(f"Loaded cached data: {val_data.shape}, labels: {val_labels.shape}")
# else:
#     print("No cached data found, loading from original files...")
#     # 只加载验证集数据
#     dataset_dict = load_multiclass_data_from_dirs(
#         DATA_DIRS, 
#         apply_pca=False,  # 先不用PCA，保留原始特征
#         pca_variance_threshold=PCA_VARIANCE_RATIO,
#         norm=False  # 后续处理会规范化
#     )
#     # 只提取验证集数据
#     val_data = dataset_dict['val_samples']
#     val_labels = dataset_dict['val_labels']
    
#     # 保存数据到检查点
#     checkpoint_mgr.save_checkpoint('data_loading', {
#         'val_data': val_data,
#         'val_labels': val_labels
#     }, data_params)
#     print(f"Saved data to checkpoint: {val_data.shape}, labels: {val_labels.shape}")

# print("\nStarting separability analysis...")

# # 询问要运行哪种分析
# analysis_choice = input("Which analysis do you want to run? (1: Raw features, 2: PCA features, 3: Both, 4: None): ")

# # 运行原始特征分析
# if analysis_choice in ['1', '3']:
#     print("\n" + "="*80)
#     print("RUNNING ANALYSIS WITH RAW FEATURES")
#     print("="*80)
#     raw_results = run_complete_separability_analysis(
#         val_data, val_labels, 
#         pca_enabled=False, 
#         save_path=os.path.join(SAVE_PATH, 'raw_features')
#     )

# # 运行PCA特征分析    
# if analysis_choice in ['2', '3']:
#     print("\n" + "="*80)
#     print("RUNNING ANALYSIS WITH PCA FEATURES")
#     print("="*80)
#     pca_results = run_complete_separability_analysis(
#         val_data, val_labels, 
#         pca_enabled=True, 
#         save_path=os.path.join(SAVE_PATH, 'pca_features')
#     )

# # 如果两种分析都完成，进行比较
# if analysis_choice == '3':
#     # 检查是否存在比较结果检查点
#     compare_results = checkpoint_mgr.load_checkpoint('comparative_analysis', {})
    
#     if compare_results is None:
#         print("\n" + "="*80)
#         print("COMPARATIVE ANALYSIS: RAW vs. PCA FEATURES")
#         print("="*80)
        
#         # 比较Fisher Ratio
#         print("\nFisher Ratio Comparison (higher is better):")
#         groups = list(raw_results['separability'].keys())
#         x = np.arange(len(groups))
#         width = 0.35
        
#         plt.figure(figsize=(12, 6))
#         raw_fisher = [raw_results['separability'][g]['fisher_ratio'] for g in groups]
#         pca_fisher = [pca_results['separability'][g]['fisher_ratio'] for g in groups]
        
#         plt.bar(x - width/2, raw_fisher, width, label='Raw Features')
#         plt.bar(x + width/2, pca_fisher, width, label='PCA Features')
        
#         plt.xlabel('Feature Groups')
#         plt.ylabel('Fisher Ratio')
#         plt.title('Fisher Ratio Comparison: Raw vs. PCA Features')
#         plt.xticks(x, groups)
#         plt.legend()
#         plt.grid(axis='y')
#         plt.savefig(os.path.join(SAVE_PATH, 'fisher_ratio_comparison.png'))
#         plt.show()
        
#         # 比较分类器性能
#         print("\nClassifier Performance Comparison (higher is better):")
#         raw_clf = [raw_results['classifier_performance'][g]['overall'] for g in groups]
#         pca_clf = [pca_results['classifier_performance'][g]['overall'] for g in groups]
        
#         plt.figure(figsize=(12, 6))
#         plt.bar(x - width/2, raw_clf, width, label='Raw Features')
#         plt.bar(x + width/2, pca_clf, width, label='PCA Features')
        
#         plt.xlabel('Feature Groups')
#         plt.ylabel('Average Classifier Performance')
#         plt.title('Classifier Performance Comparison: Raw vs. PCA Features')
#         plt.xticks(x, groups)
#         plt.legend()
#         plt.grid(axis='y')
#         plt.savefig(os.path.join(SAVE_PATH, 'classifier_performance_comparison.png'))
#         plt.show()
        
#         # 比较聚类一致性
#         print("\nClustering Consistency Comparison (higher is better):")
#         raw_consistency = list(raw_results['consistency_scores'].values())
#         pca_consistency = list(pca_results['consistency_scores'].values())
#         consistency_keys = list(raw_results['consistency_scores'].keys())
        
#         plt.figure(figsize=(12, 6))
#         x = np.arange(len(consistency_keys))
#         plt.bar(x - width/2, raw_consistency, width, label='Raw Features')
#         plt.bar(x + width/2, pca_consistency, width, label='PCA Features')
        
#         plt.xlabel('Feature Group + Clustering Method')
#         plt.ylabel('Consistency with Big Classes')
#         plt.title('Clustering Consistency Comparison: Raw vs. PCA Features')
#         plt.xticks(x, consistency_keys, rotation=45, ha='right')
#         plt.legend()
#         plt.grid(axis='y')
#         plt.tight_layout()
#         plt.savefig(os.path.join(SAVE_PATH, 'clustering_consistency_comparison.png'))
#         plt.show()
        
#         # 打印总结报告
#         print("\nSUMMARY REPORT")
#         print("="*80)
#         print("\nOverall dataset separability assessment:")
        
#         # 计算平均Fisher比率
#         avg_raw_fisher = np.mean(raw_fisher)
#         avg_pca_fisher = np.mean(pca_fisher)
#         print(f"Average Fisher ratio (raw features): {avg_raw_fisher:.4f}")
#         print(f"Average Fisher ratio (PCA features): {avg_pca_fisher:.4f}")
        
#         # 计算平均分类器性能
#         avg_raw_clf = np.mean(raw_clf)
#         avg_pca_clf = np.mean(pca_clf)
#         print(f"Average classifier performance (raw features): {avg_raw_clf:.4f}")
#         print(f"Average classifier performance (PCA features): {avg_pca_clf:.4f}")
        
#         # 计算平均聚类一致性
#         avg_raw_consistency = np.mean(raw_consistency)
#         avg_pca_consistency = np.mean(pca_consistency)
#         print(f"Average clustering consistency (raw features): {avg_raw_consistency:.4f}")
#         print(f"Average clustering consistency (PCA features): {avg_pca_consistency:.4f}")
        
#         # 理论上限估计
#         print("\nTheoretical upper bounds for classification performance:")
#         # 基于分类器、fisher ratio和聚类一致性，估计理论上限
#         best_clf_perf = max(avg_raw_clf, avg_pca_clf)
#         best_fisher = max(avg_raw_fisher, avg_pca_fisher)
#         best_consistency = max(avg_raw_consistency, avg_pca_consistency)
        
#         # 粗略估计理论上限，考虑类别平衡和fisher ratio
#         theoretical_upper_bound = min(1.0, best_clf_perf * 1.2)  # 假设当前性能可提高20%
#         print(f"Estimated theoretical upper bound (based on current analysis): {theoretical_upper_bound:.4f}")
        
#         # 解释低性能的可能原因
#         print("\nPossible reasons for limited separability:")
#         if avg_raw_fisher < 2.0 and avg_pca_fisher < 2.0:
#             print("- Low Fisher ratio indicates poor class separation in feature space")
        
#         if best_consistency < 0.7:
#             print("- Low clustering consistency suggests intrinsic class overlap")
        
#         if best_clf_perf < 0.7:
#             print("- Classifier performance indicates limitations in feature discriminative power")
        
#         # 总结建议
#         print("\nRECOMMENDATIONS:")
#         if avg_pca_fisher > avg_raw_fisher and avg_pca_clf > avg_raw_clf:
#             print("- Use PCA features for better separability")
#         else:
#             print("- Use raw features for better separability")
        
#         # 哪个特征组最有用
#         best_group_raw = groups[np.argmax(raw_clf)]
#         best_group_pca = groups[np.argmax(pca_clf)]
#         print(f"- Most discriminative feature group (raw): {best_group_raw}")
#         print(f"- Most discriminative feature group (PCA): {best_group_pca}")
        
#         # 是否需要更多特征
#         if theoretical_upper_bound < 0.8:
#             print("- Consider collecting additional features or modalities")
#             print("- Current features may be insufficient for high-performance classification")
        
#         # 修改当前dice score期望
#         current_dice = 0.60  # 当前dice score
#         print(f"\nCurrent Dice score: {current_dice:.2f}")
#         print(f"Estimated upper bound: {theoretical_upper_bound:.2f}")
        
#         if theoretical_upper_bound - current_dice < 0.1:
#             print("The current Dice score is already close to the estimated upper bound.")
#             print("Significant improvement may require fundamental changes in feature collection or methodology.")
#         else:
#             print(f"There is potential for improvement of up to {(theoretical_upper_bound - current_dice)*100:.1f}% in Dice score.")
#             print("Consider exploring advanced methods or feature combinations.")
        
#         # 保存比较结果
#         compare_results = {
#             'raw_fisher': raw_fisher,
#             'pca_fisher': pca_fisher,
#             'raw_clf': raw_clf,
#             'pca_clf': pca_clf,
#             'raw_consistency': raw_consistency,
#             'pca_consistency': pca_consistency,
#             'theoretical_upper_bound': theoretical_upper_bound
#         }
#         checkpoint_mgr.save_checkpoint('comparative_analysis', compare_results)
#     else:
#         print("\nLoading comparative analysis from checkpoint...")
#         # 从检查点加载比较结果
#         raw_fisher = compare_results['raw_fisher']
#         pca_fisher = compare_results['pca_fisher']
#         raw_clf = compare_results['raw_clf']
#         pca_clf = compare_results['pca_clf']
#         raw_consistency = compare_results['raw_consistency']
#         pca_consistency = compare_results['pca_consistency']
#         theoretical_upper_bound = compare_results['theoretical_upper_bound']
        
#         print(f"\nSummary from previous analysis:")
#         print(f"- Theoretical upper bound: {theoretical_upper_bound:.4f}")
#         print(f"- Average Fisher ratio (raw features): {np.mean(raw_fisher):.4f}")
#         print(f"- Average Fisher ratio (PCA features): {np.mean(pca_fisher):.4f}")
#         print(f"- Average classifier performance (raw features): {np.mean(raw_clf):.4f}")
#         print(f"- Average classifier performance (PCA features): {np.mean(pca_clf):.4f}")

# print("\nAnalysis complete. Checkpoint progress:")
# # 显示当前进度
# show_analysis_progress()

# # 添加简单的分析报告生成功能
# def generate_analysis_report():
#     """生成分析报告并保存到文件"""
#     report_path = os.path.join(SAVE_PATH, f"analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    
#     with open(report_path, 'w') as f:
#         f.write("脑体素数据可分性分析报告\n")
#         f.write("=========================\n\n")
#         f.write(f"分析时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
#         # 如果存在比较结果，添加比较信息
#         compare_results = checkpoint_mgr.load_checkpoint('comparative_analysis', {})
#         if compare_results is not None:
#             f.write("特征比较总结\n")
#             f.write("--------------\n")
#             raw_fisher = compare_results['raw_fisher']
#             pca_fisher = compare_results['pca_fisher']
#             raw_clf = compare_results['raw_clf']
#             pca_clf = compare_results['pca_clf']
            
#             f.write(f"理论上限: {compare_results['theoretical_upper_bound']:.4f}\n")
#             f.write(f"平均Fisher比率 (原始特征): {np.mean(raw_fisher):.4f}\n")
#             f.write(f"平均Fisher比率 (PCA特征): {np.mean(pca_fisher):.4f}\n")
#             f.write(f"平均分类器性能 (原始特征): {np.mean(raw_clf):.4f}\n")
#             f.write(f"平均分类器性能 (PCA特征): {np.mean(pca_clf):.4f}\n\n")
            
#             f.write("建议使用: ")
#             if np.mean(pca_fisher) > np.mean(raw_fisher) and np.mean(pca_clf) > np.mean(raw_clf):
#                 f.write("PCA特征\n\n")
#             else:
#                 f.write("原始特征\n\n")
            
#             # 当前性能与理论上限对比
#             current_dice = 0.60
#             f.write(f"当前Dice分数: {current_dice:.2f}\n")
#             f.write(f"估计上限: {compare_results['theoretical_upper_bound']:.2f}\n")
            
#             if compare_results['theoretical_upper_bound'] - current_dice < 0.1:
#                 f.write("当前Dice分数已接近估计上限。显著改进可能需要在特征收集或方法论上进行根本性变革。\n\n")
#             else:
#                 f.write(f"Dice分数有提升空间，最多可提高约{(compare_results['theoretical_upper_bound'] - current_dice)*100:.1f}%。\n")
#                 f.write("考虑探索高级方法或特征组合。\n\n")
        
#         # 添加每个特征组的详细分析
#         raw_results = checkpoint_mgr.load_checkpoint('complete_analysis', {'pca_enabled': False})
#         if raw_results is not None:
#             f.write("原始特征分析\n")
#             f.write("--------------\n")
#             for group_name, group_data in raw_results['separability'].items():
#                 f.write(f"{group_name} 特征组:\n")
#                 f.write(f"  - Fisher比率: {group_data['fisher_ratio']:.4f}\n")
#                 f.write(f"  - 广义判别值: {group_data['gdv']:.4f}\n")
#                 f.write(f"  - Hopkins统计量: {group_data['hopkins']:.4f}\n\n")
        
#         pca_results = checkpoint_mgr.load_checkpoint('complete_analysis', {'pca_enabled': True})
#         if pca_results is not None:
#             f.write("PCA特征分析\n")
#             f.write("--------------\n")
#             for group_name, group_data in pca_results['separability'].items():
#                 f.write(f"{group_name} 特征组:\n")
#                 f.write(f"  - Fisher比率: {group_data['fisher_ratio']:.4f}\n")
#                 f.write(f"  - 广义判别值: {group_data['gdv']:.4f}\n")
#                 f.write(f"  - Hopkins统计量: {group_data['hopkins']:.4f}\n\n")
        
#         f.write("分析完成。\n")
    
#     print(f"\n分析报告已保存到: {report_path}")
#     return report_path

# # 询问是否生成报告
# if analysis_choice in ['1', '2', '3']:
#     report_choice = input("是否生成分析报告? (y/n): ")
#     if report_choice.lower() == 'y':
#         report_path = generate_analysis_report()

# print("\n分析任务全部完成!")

In [ ]:
# 单元格: 修改后的主执行单元格，仅处理验证集
# 设置data_dirs根据你的数据路径
DATA_DIRS = {
    'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
    'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
    'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"
}

# 创建可分析路径
os.makedirs(SAVE_PATH, exist_ok=True)

# 分析进度管理
def show_analysis_progress():
    """显示当前分析进度"""
    progress = checkpoint_mgr.get_progress()
    if not progress:
        print("No analysis has been started yet.")
        return
    
    print("\nCurrent Analysis Progress:")
    print("-" * 50)
    for step, info in progress.items():
        status = "✓ Completed" if info.get('completed', False) else "⚠ Incomplete"
        timestamp = info.get('timestamp', 'N/A')
        print(f"{step:30s}: {status} ({timestamp})")
    print("-" * 50)

# 是否从头开始分析
def reset_analysis():
    """重置所有分析进度"""
    confirmation = input("Are you sure you want to reset all analysis progress? (y/n): ")
    if confirmation.lower() == 'y':
        checkpoint_mgr.reset_progress()
        print("All analysis progress has been reset.")
    else:
        print("Reset cancelled.")

# 显示当前进度
show_analysis_progress()

# 询问是否重置
reset_option = input("Do you want to reset all analysis and start fresh? (y/n): ")
if reset_option.lower() == 'y':
    reset_analysis()

# 只加载验证集数据
val_params = {'data_dir': DATA_DIRS['val_dir']}
cached_val_data = checkpoint_mgr.load_checkpoint('val_data_loading', val_params)

if cached_val_data is not None:
    print("Loading validation dataset from checkpoint...")
    val_data = cached_val_data['val_data']
    val_labels = cached_val_data['val_labels']
    print(f"Loaded cached data: {val_data.shape}, labels: {val_labels.shape}")
else:
    print("No cached validation data found, loading from original files...")
    # 修改函数为仅加载验证集数据
    def load_validation_data(val_dir, feature_dim=341, num_class=102):
        """
        只加载验证集数据
        
        Parameters:
            val_dir: 验证集目录
            feature_dim: 特征维度
            num_class: 类别数量
            
        Returns:
            val_data: 验证集特征
            val_labels: 验证集标签
        """
        print(f"Loading validation data from: {val_dir}")
        
        # 获取所有体素文件
        val_files = glob.glob(os.path.join(val_dir, "label_*_count_*_voxels.npy"))
        print(f"Found {len(val_files)} validation files")
        
        val_data_all = []
        val_labels_all = []
        
        # 处理验证集
        for file_path in val_files:
            filename = os.path.basename(file_path)
            parts = filename.split('_')
            if len(parts) >= 4 and parts[0] == 'label':
                try:
                    label_id = int(parts[1])
                    if 0 <= label_id < num_class:
                        print(f"Loading feature data: {filename}")
                        voxels = np.load(file_path)
                        labels = np.full(len(voxels), label_id)
                        val_data_all.append(voxels)
                        val_labels_all.append(labels)
                    else:
                        print(f"Warning: Skipping label {label_id}, out of range [0, {num_class-1}]")
                except ValueError:
                    print(f"Warning: Could not extract label ID from {filename}")
        
        # 合并所有数据
        if val_data_all:
            val_data = np.vstack(val_data_all)
            val_labels = np.concatenate(val_labels_all)
        else:
            raise ValueError("No valid validation data found")
        
        # 打印数据范围和标签信息
        print(f"Validation data range: {np.min(val_data)} to {np.max(val_data)}")
        print(f"Validation labels range: {np.min(val_labels)} to {np.max(val_labels)}")
        
        # 统计各类别样本数
        class_counts = np.zeros(num_class, dtype=int)
        for i in range(num_class):
            class_counts[i] = np.sum(val_labels == i)
        
        # 打印类别分布（仅打印有样本的类别）
        print("\nClass distribution in validation set:")
        print(f"{'Class ID':^10}{'Count':^10}{'Percentage':^15}")
        print("-" * 35)
        
        total_samples = len(val_labels)
        for i in range(num_class):
            if class_counts[i] > 0:
                percentage = class_counts[i] / total_samples * 100
                print(f"{i:^10}{class_counts[i]:^10}{percentage:.2f}%:^15}")
        
        print("-" * 35)
        print(f"{'Total':^10}{total_samples:^10}{'100.00%':^15}")
        
        return val_data, val_labels
    
    # 加载验证集数据
    val_data, val_labels = load_validation_data(DATA_DIRS['val_dir'], FEATURE_DIM, NUM_CLASS)
    
    # 保存数据到检查点
    checkpoint_mgr.save_checkpoint('val_data_loading', {
        'val_data': val_data,
        'val_labels': val_labels
    }, val_params)
    print(f"Saved validation data to checkpoint: {val_data.shape}, labels: {val_labels.shape}")

print("\nStarting separability analysis...")

# 询问要运行哪种分析
analysis_choice = input("Which analysis do you want to run? (1: Raw features, 2: PCA features, 3: Both, 4: None): ")

# 运行原始特征分析
if analysis_choice in ['1', '3']:
    print("\n" + "="*80)
    print("RUNNING ANALYSIS WITH RAW FEATURES")
    print("="*80)
    raw_results = run_complete_separability_analysis(
        val_data, val_labels, 
        pca_enabled=False, 
        save_path=os.path.join(SAVE_PATH, 'raw_features')
    )

# 运行PCA特征分析    
if analysis_choice in ['2', '3']:
    print("\n" + "="*80)
    print("RUNNING ANALYSIS WITH PCA FEATURES")
    print("="*80)
    pca_results = run_complete_separability_analysis(
        val_data, val_labels, 
        pca_enabled=True, 
        save_path=os.path.join(SAVE_PATH, 'pca_features')
    )

# 如果两种分析都完成，进行比较
if analysis_choice == '3':
    # 检查是否存在比较结果检查点
    compare_results = checkpoint_mgr.load_checkpoint('comparative_analysis', {})
    
    if compare_results is None:
        print("\n" + "="*80)
        print("COMPARATIVE ANALYSIS: RAW vs. PCA FEATURES")
        print("="*80)
        
        # 比较Fisher Ratio
        print("\nFisher Ratio Comparison (higher is better):")
        groups = list(raw_results['separability'].keys())
        x = np.arange(len(groups))
        width = 0.35
        
        plt.figure(figsize=(12, 6))
        raw_fisher = [raw_results['separability'][g]['fisher_ratio'] for g in groups]
        pca_fisher = [pca_results['separability'][g]['fisher_ratio'] for g in groups]
        
        plt.bar(x - width/2, raw_fisher, width, label='Raw Features')
        plt.bar(x + width/2, pca_fisher, width, label='PCA Features')
        
        plt.xlabel('Feature Groups')
        plt.ylabel('Fisher Ratio')
        plt.title('Fisher Ratio Comparison: Raw vs. PCA Features')
        plt.xticks(x, groups)
        plt.legend()
        plt.grid(axis='y')
        plt.savefig(os.path.join(SAVE_PATH, 'fisher_ratio_comparison.png'))
        plt.show()
        
        # 比较GDV（新的论文实现版本）
        print("\nGDV Comparison (lower/more negative is better):")
        raw_gdv = [raw_results['separability'][g]['gdv'] for g in groups]
        pca_gdv = [pca_results['separability'][g]['gdv'] for g in groups]
        
        plt.figure(figsize=(12, 6))
        plt.bar(x - width/2, raw_gdv, width, label='Raw Features')
        plt.bar(x + width/2, pca_gdv, width, label='PCA Features')
        
        plt.xlabel('Feature Groups')
        plt.ylabel('GDV (lower is better)')
        plt.title('Generalized Discrimination Value Comparison')
        plt.xticks(x, groups)
        plt.legend()
        plt.grid(axis='y')
        plt.savefig(os.path.join(SAVE_PATH, 'gdv_comparison.png'))
        plt.show()
        
        # 比较分类器性能
        print("\nClassifier Performance Comparison (higher is better):")
        raw_clf = [raw_results['classifier_performance'][g]['overall'] for g in groups]
        pca_clf = [pca_results['classifier_performance'][g]['overall'] for g in groups]
        
        plt.figure(figsize=(12, 6))
        plt.bar(x - width/2, raw_clf, width, label='Raw Features')
        plt.bar(x + width/2, pca_clf, width, label='PCA Features')
        
        plt.xlabel('Feature Groups')
        plt.ylabel('Average Classifier Performance')
        plt.title('Classifier Performance Comparison')
        plt.xticks(x, groups)
        plt.legend()
        plt.grid(axis='y')
        plt.savefig(os.path.join(SAVE_PATH, 'classifier_performance_comparison.png'))
        plt.show()
        
        # 比较聚类一致性
        print("\nClustering Consistency Comparison (higher is better):")
        raw_consistency = list(raw_results['consistency_scores'].values())
        pca_consistency = list(pca_results['consistency_scores'].values())
        consistency_keys = list(raw_results['consistency_scores'].keys())
        
        plt.figure(figsize=(12, 6))
        x = np.arange(len(consistency_keys))
        plt.bar(x - width/2, raw_consistency, width, label='Raw Features')
        plt.bar(x + width/2, pca_consistency, width, label='PCA Features')
        
        plt.xlabel('Feature Group + Clustering Method')
        plt.ylabel('Consistency with Big Classes')
        plt.title('Clustering Consistency Comparison')
        plt.xticks(x, consistency_keys, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y')
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_PATH, 'clustering_consistency_comparison.png'))
        plt.show()
        
        # 打印总结报告
        print("\nSUMMARY REPORT (VALIDATION SET ONLY)")
        print("="*80)
        print("\nOverall validation dataset separability assessment:")
        
        # 计算平均Fisher比率
        avg_raw_fisher = np.mean(raw_fisher)
        avg_pca_fisher = np.mean(pca_fisher)
        print(f"Average Fisher ratio (raw features): {avg_raw_fisher:.4f}")
        print(f"Average Fisher ratio (PCA features): {avg_pca_fisher:.4f}")
        
        # 计算平均GDV
        avg_raw_gdv = np.mean(raw_gdv)
        avg_pca_gdv = np.mean(pca_gdv)
        print(f"Average GDV (raw features): {avg_raw_gdv:.4f}")
        print(f"Average GDV (PCA features): {avg_pca_gdv:.4f}")
        
        # 计算平均分类器性能
        avg_raw_clf = np.mean(raw_clf)
        avg_pca_clf = np.mean(pca_clf)
        print(f"Average classifier performance (raw features): {avg_raw_clf:.4f}")
        print(f"Average classifier performance (PCA features): {avg_pca_clf:.4f}")
        
        # 计算平均聚类一致性
        avg_raw_consistency = np.mean(raw_consistency)
        avg_pca_consistency = np.mean(pca_consistency)
        print(f"Average clustering consistency (raw features): {avg_raw_consistency:.4f}")
        print(f"Average clustering consistency (PCA features): {avg_pca_consistency:.4f}")
        
        # 理论上限估计
        print("\nTheoretical upper bounds for classification performance:")
        # 基于分类器、fisher ratio和聚类一致性，估计理论上限
        best_clf_perf = max(avg_raw_clf, avg_pca_clf)
        best_fisher = max(avg_raw_fisher, avg_pca_fisher)
        best_consistency = max(avg_raw_consistency, avg_pca_consistency)
        
        # 粗略估计理论上限，考虑类别平衡和fisher ratio
        theoretical_upper_bound = min(1.0, best_clf_perf * 1.2)  # 假设当前性能可提高20%
        print(f"Estimated theoretical upper bound (based on current analysis): {theoretical_upper_bound:.4f}")
        
        # 解释低性能的可能原因
        print("\nPossible reasons for limited separability:")
        if avg_raw_fisher < 2.0 and avg_pca_fisher < 2.0:
            print("- Low Fisher ratio indicates poor class separation in feature space")
        
        # GDV值越负越好，所以使用相反的判断条件
        if avg_raw_gdv > -0.1 and avg_pca_gdv > -0.1:
            print("- GDV values close to zero indicate weak class separation")
        
        if best_consistency < 0.7:
            print("- Low clustering consistency suggests intrinsic class overlap")
        
        if best_clf_perf < 0.7:
            print("- Classifier performance indicates limitations in feature discriminative power")
        
        # 总结建议
        print("\nRECOMMENDATIONS:")
        if avg_pca_fisher > avg_raw_fisher and avg_pca_clf > avg_raw_clf:
            print("- Use PCA features for better separability")
        else:
            print("- Use raw features for better separability")
        
        # 哪个特征组最有用
        best_group_raw = groups[np.argmax(raw_clf)]
        best_group_pca = groups[np.argmax(pca_clf)]
        print(f"- Most discriminative feature group (raw): {best_group_raw}")
        print(f"- Most discriminative feature group (PCA): {best_group_pca}")
        
        # 是否需要更多特征
        if theoretical_upper_bound < 0.8:
            print("- Consider collecting additional features or modalities")
            print("- Current features may be insufficient for high-performance classification")
        
        # 修改当前dice score期望
        current_dice = 0.60  # 当前dice score
        print(f"\nCurrent Dice score: {current_dice:.2f}")
        print(f"Estimated upper bound: {theoretical_upper_bound:.2f}")
        
        if theoretical_upper_bound - current_dice < 0.1:
            print("The current Dice score is already close to the estimated upper bound.")
            print("Significant improvement may require fundamental changes in feature collection or methodology.")
        else:
            print(f"There is potential for improvement of up to {(theoretical_upper_bound - current_dice)*100:.1f}% in Dice score.")
            print("Consider exploring advanced methods or feature combinations.")
        
        # 保存比较结果
        compare_results = {
            'raw_fisher': raw_fisher,
            'pca_fisher': pca_fisher,
            'raw_gdv': raw_gdv,
            'pca_gdv': pca_gdv,
            'raw_clf': raw_clf,
            'pca_clf': pca_clf,
            'raw_consistency': raw_consistency,
            'pca_consistency': pca_consistency,
            'theoretical_upper_bound': theoretical_upper_bound
        }
        checkpoint_mgr.save_checkpoint('comparative_analysis', compare_results)
    else:
        print("\nLoading comparative analysis from checkpoint...")
        # 从检查点加载比较结果
        raw_fisher = compare_results['raw_fisher']
        pca_fisher = compare_results['pca_fisher']
        raw_gdv = compare_results.get('raw_gdv', [])  # 兼容旧检查点
        pca_gdv = compare_results.get('pca_gdv', [])
        raw_clf = compare_results['raw_clf']
        pca_clf = compare_results['pca_clf']
        theoretical_upper_bound = compare_results['theoretical_upper_bound']
        
        print(f"\nSummary from previous analysis (validation set):")
        print(f"- Theoretical upper bound: {theoretical_upper_bound:.4f}")
        print(f"- Average Fisher ratio (raw features): {np.mean(raw_fisher):.4f}")
        print(f"- Average Fisher ratio (PCA features): {np.mean(pca_fisher):.4f}")
        if raw_gdv and pca_gdv:
            print(f"- Average GDV (raw features): {np.mean(raw_gdv):.4f}")
            print(f"- Average GDV (PCA features): {np.mean(pca_gdv):.4f}")
        print(f"- Average classifier performance (raw features): {np.mean(raw_clf):.4f}")
        print(f"- Average classifier performance (PCA features): {np.mean(pca_clf):.4f}")

# 询问是否生成报告
if analysis_choice in ['1', '2', '3']:
    report_choice = input("Do you want to generate an analysis report? (y/n): ")
    if report_choice.lower() == 'y':
        report_path = os.path.join(SAVE_PATH, f"analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
        
        with open(report_path, 'w') as f:
            f.write("Brain Voxel Data Separability Analysis Report (Validation Set Only)\n")
            f.write("=================================================================\n\n")
            f.write(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 如果存在比较结果，添加比较信息
            compare_results = checkpoint_mgr.load_checkpoint('comparative_analysis', {})
            if compare_results is not None:
                f.write("Feature Comparison Summary\n")
                f.write("-------------------------\n")
                raw_fisher = compare_results['raw_fisher']
                pca_fisher = compare_results['pca_fisher']
                raw_clf = compare_results['raw_clf']
                pca_clf = compare_results['pca_clf']
                raw_gdv = compare_results.get('raw_gdv', [])
                pca_gdv = compare_results.get('pca_gdv', [])
                
                f.write(f"Theoretical upper bound: {compare_results['theoretical_upper_bound']:.4f}\n")
                f.write(f"Average Fisher ratio (raw features): {np.mean(raw_fisher):.4f}\n")
                f.write(f"Average Fisher ratio (PCA features): {np.mean(pca_fisher):.4f}\n")
                if raw_gdv and pca_gdv:
                    f.write(f"Average GDV (raw features): {np.mean(raw_gdv):.4f}\n")
                    f.write(f"Average GDV (PCA features): {np.mean(pca_gdv):.4f}\n")
                f.write(f"Average classifier performance (raw features): {np.mean(raw_clf):.4f}\n")
                f.write(f"Average classifier performance (PCA features): {np.mean(pca_clf):.4f}\n\n")
                
                f.write("Recommended feature type: ")
                if np.mean(pca_fisher) > np.mean(raw_fisher) and np.mean(pca_clf) > np.mean(raw_clf):
                    f.write("PCA features\n\n")
                else:
                    f.write("Raw features\n\n")
                
                # 当前性能与理论上限对比
                current_dice = 0.60
                f.write(f"Current Dice score: {current_dice:.2f}\n")
                f.write(f"Estimated upper bound: {compare_results['theoretical_upper_bound']:.2f}\n")
                
                if compare_results['theoretical_upper_bound'] - current_dice < 0.1:
                    f.write("The current Dice score is already close to the estimated upper bound.\n")
                    f.write("Significant improvement may require fundamental changes in feature collection or methodology.\n\n")
                else:
                    f.write(f"There is potential for improvement of up to {(compare_results['theoretical_upper_bound'] - current_dice)*100:.1f}% in Dice score.\n")
                    f.write("Consider exploring advanced methods or feature combinations.\n\n")
            
            f.write("Analysis complete.\n")
        
        print(f"\nAnalysis report saved to: {report_path}")

print("\nAnalysis complete. Checkpoint progress:")
show_analysis_progress()